In [ ]:

# Start the session timer before dependency setup.
import time
SESSION_START = time.time()
print("Session timer started")


In [ ]:
# Dependency preflight for the Kaggle runtime.
# Preserve the preinstalled torch, torchvision, Pillow, NumPy, pandas, and scikit-learn stack.
# In-place package upgrades can create runtime incompatibilities.


import importlib.util, importlib.metadata as md, subprocess, sys

print("Base environment versions (left untouched):")
for pkg in ["torch", "torchvision", "Pillow", "numpy", "pandas", "scikit-learn", "transformers"]:
    try:
        print(f"  {pkg:15s} {md.version(pkg)}")
    except Exception:
        print(f"  {pkg:15s} <not installed>")

# Install the pinned Transformers version only when the package is unavailable.
if importlib.util.find_spec("transformers") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers==4.56.2", "tokenizers>=0.20,<0.22", "safetensors>=0.4"
    ])

# Install the official BanglaBERT normalizer when unavailable.
if importlib.util.find_spec("normalizer") is None:
    normalizer_install = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/csebuetnlp/normalizer"],
        check=False,
    )
    if normalizer_install.returncode != 0:
        print("Bangla normalizer install failed; fallback NFC normalization will be used.")

print("Dependency preflight complete. Pillow/torchvision were not modified.")


In [ ]:
# Imports, dataset discovery, dataset fingerprints, and run configuration.
import os, re, gc, json, math, random, hashlib, shutil, warnings, unicodedata
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageFile, ImageOps
ImageFile.LOAD_TRUNCATED_IMAGES = True

try:
    _pil_probe = Image.new("RGB", (2, 2), "white")
    _pil_probe.resize((1, 1), Image.Resampling.BICUBIC)
    del _pil_probe
except Exception as e:
    raise RuntimeError(
        "Pillow is corrupted in this live Kaggle session. Start a NEW Kaggle session "
        "and run this notebook from Cell 0. Original error: " + str(e)
    ) from e

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer, AutoModel,
    CLIPVisionModel,
    get_cosine_schedule_with_warmup,
)
from sklearn.metrics import (
    f1_score, accuracy_score, confusion_matrix,
    precision_recall_fscore_support,
    average_precision_score, hamming_loss,
)

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    from normalizer import normalize as bangla_normalize
    HAVE_BANGLA_NORMALIZER = True
except Exception:
    HAVE_BANGLA_NORMALIZER = False
    bangla_normalize = None


# Publication seed selection.
# SEED_INDEX selects one of the five fixed publication seeds.
# Scientific hyperparameters remain fixed across seeds.

PUBLICATION_SEEDS = [42, 1337, 2026, 31415, 271828]
SEED_INDEX = 0
if SEED_INDEX not in range(len(PUBLICATION_SEEDS)):
    raise ValueError(f"SEED_INDEX must be 0..{len(PUBLICATION_SEEDS)-1}")
RUN_SEED = PUBLICATION_SEEDS[SEED_INDEX]
print("PUBLICATION SEED:", RUN_SEED)

EXPECTED_SHA256 = {
    "label.json": "9cb9afdd88c9a750d3eb6f36aa512dc6ad5b3067aedfac79b138b7fef64a9bb6",
    "ocr.csv": "eed35bf77f749e98d9222fd571d149b99c4c6a610e29bc5db5783d3489492c0c",
    "2_hit_removed_split_manifest.jsonl": "bbcf7e13fc0aab4ac3b5f58a2ebf8880740ed60ad7ecb2cb61910b0497d1b4f0",
}
REQUIRED_DATA_FILES = tuple(EXPECTED_SHA256.keys())

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def structurally_valid(root: Path) -> bool:
    return (
        root.is_dir()
        and all((root / name).is_file() for name in REQUIRED_DATA_FILES)
        and (root / "images").is_dir()
    )

def fingerprints_match(root: Path) -> bool:
    try:
        return all(
            sha256_file(root / name) == EXPECTED_SHA256[name]
            for name in REQUIRED_DATA_FILES
        )
    except Exception:
        return False

def resolve_dataset_root() -> Path:
    # Identify the dataset from mounted inputs using structure and SHA-256 fingerprints.
    
    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.exists():
        raise FileNotFoundError("/kaggle/input not found. Run this notebook on Kaggle.")

    candidates = []
    for label_file in kaggle_input.rglob("label.json"):
        root = label_file.parent
        if structurally_valid(root):
            candidates.append(root)

    candidates = sorted(set(candidates), key=lambda p: str(p))
    if not candidates:
        raise FileNotFoundError(
            "No attached Kaggle dataset contains label.json, ocr.csv, "
            "2_hit_removed_split_manifest.jsonl, and images/ in one folder."
        )

    matching = [root for root in candidates if fingerprints_match(root)]
    if len(matching) != 1:
        print("Structurally valid candidates:")
        for root in candidates:
            print(" -", root)
        raise RuntimeError(
            f"Expected exactly one M3Offend fingerprint match, found {len(matching)}. "
            "Attach only the canonical release or remove ambiguous copies."
        )
    print("[DATA] Canonical M3Offend automatically resolved:", matching[0])
    return matching[0]

DATA_ROOT = resolve_dataset_root()
LABEL_FILE = DATA_ROOT / "label.json"
OCR_FILE = DATA_ROOT / "ocr.csv"
SPLIT_FILE = DATA_ROOT / "2_hit_removed_split_manifest.jsonl"
IMAGE_DIR = DATA_ROOT / "images"

OUT = Path("/kaggle/working/M3Offend_9318_BanglaBERT_CLIP_FacetGate_ALL5_FINAL_LOCKED_TEST")
OUT.mkdir(parents=True, exist_ok=True)
CKPT = OUT / "checkpoints"
CKPT.mkdir(exist_ok=True)

CFG = {
    "notebook_revision": "FacetGate_v1_0_1_controlled_BanglaBERT_binding",
    "comparison_status": "FacetGate_controlled_binding__not_strict_Frozen_Common_Gated_v1_1",
    "publication_seeds": PUBLICATION_SEEDS,
    "run_seed": RUN_SEED,

    "text_model": "csebuetnlp/banglabert",
    "vision_model": "openai/clip-vit-base-patch32",
    "text_revision": "9ce791f330578f50da6bc52b54205166fb5d1c8c",
    "vision_revision": "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268",
    "facetgate_common_protocol_sha256": "37b7d30a994fb98225d49d76cca5768a838d456abf844dc1a5d93019a0cd15aa",

    "expected_total": 9318,
    "expected_splits": {"train": 7454, "validation": 932, "test": 932},
    "expected_sha256": EXPECTED_SHA256,

    "binary_tasks": ["offensive", "vulgar", "abuse", "sarcastic"],
    "languages": ["Bangla", "English", "Banglish"],
    "targets": ["Individual", "Gender", "Political", "Social", "National", "Religious", "Others"],

    # BanglaBERT+CLIP training configuration.
    "max_text_length": 384,
    "batch_size": 4,
    "epochs": 20,
    "early_stopping_patience": 5,
    "warmup_epochs": 3,
    "image_lr": 1e-5,
    "text_lr": 2e-5,
    "fusion_lr": 5e-5,
    "weight_decay": 0.1,
    "focal_gamma": 2.0,
    "focal_alpha": 0.75,
    "dropout": 0.4,
    "aux_loss_weight": 0.3,
    "unfreeze_text_layers": 2,
    "unfreeze_vision_layers": 2,
    "gradient_clip": 1.0,
    "num_workers": 2,
    "use_rarity_sampler": False,
    "rarity_sampler_power": 0.35,
    "rarity_sampler_cap": 3.0,
    "target_weight_power": 0.5,
    "target_weight_ratio_cap": 12.0,
    "binary_weight_power": 1.0,
    "language_weight_power": 0.5,

    # Validation and threshold-calibration settings.
    "threshold_min": 0.05,
    "threshold_max": 0.95,
    "threshold_step": 0.025,
    "threshold_min_support": 5,
    "threshold_min_improvement": 0.005,

    "bootstrap_repeats": 2000,
}

with open(OUT / "resolved_config.json", "w") as f:
    json.dump(CFG, f, indent=2)

print("DATA ROOT:", DATA_ROOT)
print("OUTPUT:", OUT)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i),
          round(torch.cuda.get_device_properties(i).total_memory/1024**3, 2), "GB")

if torch.cuda.device_count() < 2:
    raise RuntimeError(
        "Select Kaggle GPU T4 x2. This notebook places BanglaBERT on GPU0 and CLIP on GPU1."
    )

In [ ]:
# Dataset integrity checks and parsing; the frozen split is used unchanged.

for p in [LABEL_FILE, OCR_FILE, SPLIT_FILE]:
    if not p.exists():
        raise FileNotFoundError(p)
    observed = sha256_file(p)
    expected = CFG["expected_sha256"][p.name]
    print(p.name, observed)
    if observed != expected:
        raise ValueError(f"Fingerprint mismatch for {p.name}; expected {expected}, observed {observed}")

with open(LABEL_FILE, encoding="utf-8") as f:
    raw_labels = json.load(f)

LANGUAGE_ALIASES = {
    "Bangla": "Bangla",
    "English": "English",
    "Banglish": "Banglish",
    "Banglish (Bangla + English)": "Banglish",  # Legacy alias normalization.
}

rows = []
for item in raw_labels:
    image = item["data"]["image"]
    result = item["annotations"][0]["result"]
    m = {r["from_name"]: list(r["value"]["choices"]) for r in result}
    row = {"image": image}

    for task in CFG["binary_tasks"]:
        if m.get(task, [None])[0] not in {"Yes", "No"}:
            raise ValueError((image, task, m.get(task)))
        row[task] = 1 if m[task][0] == "Yes" else 0

    raw_language = m["language"][0]
    if raw_language not in LANGUAGE_ALIASES:
        raise ValueError((image, "language", raw_language))
    row["language"] = LANGUAGE_ALIASES[raw_language]

    targets = m.get("target", [])
    if targets == ["None"]:
        targets = []
    illegal = set(targets) - set(CFG["targets"])
    if illegal:
        raise ValueError((image, "target", illegal))
    row["target_list"] = targets
    for t in CFG["targets"]:
        row[f"target__{t}"] = int(t in targets)
    rows.append(row)

labels = pd.DataFrame(rows)
labels.insert(0, "official_row_index", np.arange(len(labels), dtype=np.int64))

ocr = pd.read_csv(OCR_FILE)
required_ocr = {"source_image_name", "ocr_text"}
if not required_ocr.issubset(ocr.columns):
    raise ValueError(f"OCR columns missing: {required_ocr - set(ocr.columns)}")
ocr = ocr.rename(columns={"source_image_name": "image"})
ocr["ocr_text"] = ocr["ocr_text"].fillna("").astype(str)

split = pd.read_json(SPLIT_FILE, lines=True)
if set(split.columns) != {"image", "split"}:
    raise ValueError(f"Unexpected split columns: {split.columns.tolist()}")

for name, frame in [("labels", labels), ("ocr", ocr), ("split", split)]:
    if frame["image"].duplicated().any():
        dup = frame.loc[frame["image"].duplicated(False), "image"].tolist()[:10]
        raise ValueError(f"Duplicate IDs in {name}: {dup}")
    if len(frame) != CFG["expected_total"]:
        raise ValueError(f"{name}: {len(frame)} != {CFG['expected_total']}")

if set(labels.image) != set(ocr.image) or set(labels.image) != set(split.image):
    raise ValueError("Label/OCR/split image-ID sets are not identical")

full = labels.merge(ocr[["image", "ocr_text"]], on="image", validate="one_to_one")
full = full.merge(split, on="image", validate="one_to_one")

split_counts = full["split"].value_counts().to_dict()
print("split counts:", split_counts)
if split_counts != CFG["expected_splits"]:
    raise ValueError((split_counts, CFG["expected_splits"]))

missing_images = [x for x in full.image if not (IMAGE_DIR / x).exists()]
if missing_images:
    raise FileNotFoundError(f"Missing {len(missing_images)} images; first 20: {missing_images[:20]}")

# Empty cleaned OCR remains valid under the frozen OCR policy.
blank_ocr_n = int((full.ocr_text.str.strip() == "").sum())
print("blank cleaned-OCR rows (valid):", blank_ocr_n)

# Audit label consistency without imposing prediction-time constraints.
target_cols = [f"target__{t}" for t in CFG["targets"]]
invalid_abuse = full[(full.abuse == 1) & (full[target_cols].sum(axis=1) == 0)]
print("gold abuse=yes + empty target audit count:", len(invalid_abuse))
if len(invalid_abuse):
    invalid_abuse.to_csv(OUT / "gold_abuse_yes_empty_target_audit.csv", index=False)

# Check for exact-image leakage across splits.
print("Checking exact image hashes across frozen partitions ...")
hash_to_rows = defaultdict(list)
for image, sp in tqdm(full[["image", "split"]].itertuples(index=False), total=len(full)):
    h = sha256_file(IMAGE_DIR / image)
    hash_to_rows[h].append((image, sp))

leaks = []
for h, members in hash_to_rows.items():
    if len({sp for _, sp in members}) > 1:
        leaks.append({"sha256": h, "members": members})
if leaks:
    with open(OUT / "cross_split_exact_duplicate_leaks.json", "w") as f:
        json.dump(leaks, f, indent=2)
    raise ValueError(f"Found {len(leaks)} exact-image groups crossing splits. Stop before training.")
print("Exact-image cross-split leakage: 0 groups")

audit_rows = []
for sp in ["train", "validation", "test"]:
    d = full[full.split == sp]
    for task in CFG["binary_tasks"]:
        audit_rows.append({"split": sp, "label": task, "state": "Yes", "count": int(d[task].sum())})
    for lang in CFG["languages"]:
        audit_rows.append({"split": sp, "label": "language", "state": lang, "count": int((d.language == lang).sum())})
    for t in CFG["targets"]:
        audit_rows.append({"split": sp, "label": "target", "state": t, "count": int(d[f"target__{t}"].sum())})
pd.DataFrame(audit_rows).to_csv(OUT / "split_distribution_audit.csv", index=False)

print(full.head(2))
print("Integrity audit PASSED")


In [ ]:

# Text normalization preserves lexical content and punctuation.
_bn_re = re.compile(r"[ঀ-৿]")

def normalize_ocr(text):
    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text).strip()
    if HAVE_BANGLA_NORMALIZER and _bn_re.search(text):
        try:
            text = bangla_normalize(text)
        except Exception:
            pass
    return re.sub(r"\s+", " ", text).strip()

full["model_text"] = full["ocr_text"].map(normalize_ocr)
print("Official BanglaBERT normalizer available:", HAVE_BANGLA_NORMALIZER)
print(full[["ocr_text", "model_text"]].head(3).to_string(index=False))

lang_to_id = {x:i for i,x in enumerate(CFG["languages"])}
full["language_id"] = full.language.map(lang_to_id)

# Preserve the frozen row order within each partition.
train_df = full[full.split == "train"].reset_index(drop=True)
val_df = full[full.split == "validation"].reset_index(drop=True)
test_df = full[full.split == "test"].reset_index(drop=True)

print(len(train_df), len(val_df), len(test_df))


In [ ]:

# Tokenization, CLIP ViT-B/32 preprocessing, and dataset definition.
text_tokenizer = AutoTokenizer.from_pretrained(
    CFG["text_model"], revision=CFG["text_revision"]
)

# CLIP ViT-B/32 preprocessing:
# RGB conversion; resize shortest edge to 224 with bicubic interpolation;
# center crop to 224x224; scale pixel values by 1/255;
# normalize with CLIP mean and standard deviation.
# Direct preprocessing avoids additional torchvision processor dependencies.
CLIP_SIZE = 224
CLIP_MEAN = np.asarray([0.48145466, 0.4578275, 0.40821073], dtype=np.float32)
CLIP_STD  = np.asarray([0.26862954, 0.26130258, 0.27577711], dtype=np.float32)

def clip_preprocess_pil(image: Image.Image) -> torch.Tensor:
    image = ImageOps.exif_transpose(image).convert("RGB")
    w, h = image.size
    if w <= 0 or h <= 0:
        raise ValueError(f"Invalid image size: {(w, h)}")

    scale = CLIP_SIZE / min(w, h)
    new_w = max(CLIP_SIZE, int(w * scale))
    new_h = max(CLIP_SIZE, int(h * scale))
    image = image.resize((new_w, new_h), resample=Image.Resampling.BICUBIC)

    left = (new_w - CLIP_SIZE) // 2
    top = (new_h - CLIP_SIZE) // 2
    image = image.crop((left, top, left + CLIP_SIZE, top + CLIP_SIZE))

    arr = np.asarray(image, dtype=np.float32) / 255.0
    if arr.shape != (CLIP_SIZE, CLIP_SIZE, 3):
        raise ValueError(f"Unexpected RGB array shape: {arr.shape}")
    arr = (arr - CLIP_MEAN[None, None, :]) / CLIP_STD[None, None, :]
    arr = np.ascontiguousarray(arr.transpose(2, 0, 1))
    return torch.from_numpy(arr)

# Preprocessing sanity check.
_test_img = Image.new("RGB", (320, 180), (127, 127, 127))
_test_pixel = clip_preprocess_pil(_test_img)
assert _test_pixel.shape == (3, 224, 224)
assert _test_pixel.dtype == torch.float32
print("Manual CLIP preprocessing sanity check:", tuple(_test_pixel.shape), _test_pixel.dtype)
del _test_img, _test_pixel

class MemeDataset(Dataset):
    def __init__(self, frame):
        self.df = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        tok = text_tokenizer(
            r.model_text,
            max_length=CFG["max_text_length"],
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        with Image.open(IMAGE_DIR / r.image) as im:
            pixel = clip_preprocess_pil(im)
        labels = {
            task: torch.tensor(int(r[task]), dtype=torch.long) for task in CFG["binary_tasks"]
        }
        labels["language"] = torch.tensor(int(r.language_id), dtype=torch.long)
        labels["target"] = torch.tensor([int(r[f"target__{t}"]) for t in CFG["targets"]], dtype=torch.long)
        return {
            "image": r.image,
            "input_ids": tok["input_ids"][0],
            "attention_mask": tok["attention_mask"][0],
            "pixel_values": pixel,
            "labels": labels,
        }

train_ds, val_ds, test_ds = MemeDataset(train_df), MemeDataset(val_df), MemeDataset(test_df)

# Optional capped rarity sampler computed from training labels only.
def build_sample_weights(frame):
    n = len(frame)
    rarity = np.ones(n, dtype=np.float64)
    power = CFG["rarity_sampler_power"]
    for task in CFG["binary_tasks"]:
        pos = max(1, int(frame[task].sum()))
        neg = max(1, n - pos)
        ratio = neg / pos
        rarity += frame[task].to_numpy() * (ratio ** power - 1.0)
    for t in CFG["targets"]:
        col = f"target__{t}"
        pos = max(1, int(frame[col].sum()))
        neg = max(1, n - pos)
        ratio = neg / pos
        rarity += frame[col].to_numpy() * 0.35 * (ratio ** power - 1.0)
    lang_counts = frame.language.value_counts()
    maxc = lang_counts.max()
    for lang in CFG["languages"]:
        ratio = maxc / max(1, lang_counts.get(lang, 1))
        rarity += (frame.language.to_numpy() == lang) * 0.15 * (ratio ** power - 1.0)
    rarity = np.clip(rarity, 1.0, CFG["rarity_sampler_cap"])
    rarity /= rarity.mean()
    return torch.tensor(rarity, dtype=torch.double)

sample_weights = build_sample_weights(train_df)
print(pd.Series(sample_weights.numpy()).describe())


In [ ]:

# Adaptive late fusion for the six-task formulation.
def inv_softplus(x):
    return math.log(math.exp(x) - 1.0)

class AdaptiveLateFusion(nn.Module):
    """Paper's temperature-scaled convex logit fusion, generalized to n independent units."""
    def __init__(self, n_units=1):
        super().__init__()
        self.n_units = n_units
        self.weight_logits = nn.Parameter(torch.zeros(n_units, 2))  # [image, text]
        self.raw_temperature = nn.Parameter(torch.full((n_units,), inv_softplus(1.0)))

    def forward(self, image_logits, text_logits):
        w = torch.softmax(self.weight_logits, dim=-1)
        tau = F.softplus(self.raw_temperature) + 1e-4
        if image_logits.ndim == 2:  # [B, C]
            return w[0,0] * (image_logits / tau[0]) + w[0,1] * (text_logits / tau[0])
        if image_logits.ndim == 3:  # [B, U, C]
            return (
                w[None,:,0,None] * (image_logits / tau[None,:,None])
                + w[None,:,1,None] * (text_logits / tau[None,:,None])
            )
        raise ValueError(image_logits.shape)

class ModalityHeads(nn.Module):
    def __init__(self, hidden_dim, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.binary = nn.ModuleDict({t: nn.Linear(hidden_dim, 2) for t in CFG["binary_tasks"]})
        self.language = nn.Linear(hidden_dim, len(CFG["languages"]))
        self.target = nn.Linear(hidden_dim, len(CFG["targets"]) * 2)
    def forward(self, h):
        h = self.dropout(h)
        out = {t: self.binary[t](h) for t in CFG["binary_tasks"]}
        out["language"] = self.language(h)
        out["target"] = self.target(h).view(-1, len(CFG["targets"]), 2)
        return out

class MemeGuardSixTask(nn.Module):
    def __init__(self, device_text="cuda:0", device_image="cuda:1"):
        super().__init__()
        self.device_text = torch.device(device_text)
        self.device_image = torch.device(device_image)

        self.text_encoder = AutoModel.from_pretrained(
            CFG["text_model"], revision=CFG["text_revision"]
        )
        self.image_encoder = CLIPVisionModel.from_pretrained(
            CFG["vision_model"], revision=CFG["vision_revision"]
        )

        # Freeze pretrained parameters before selective unfreezing.
        for p in self.text_encoder.parameters(): p.requires_grad = False
        for p in self.image_encoder.parameters(): p.requires_grad = False

        # Match the released baseline configuration by unfreezing the final two ELECTRA layers.
        text_layers = self.text_encoder.encoder.layer
        for layer in text_layers[-CFG["unfreeze_text_layers"]:]:
            for p in layer.parameters(): p.requires_grad = True

        # Match the released baseline configuration by unfreezing the final two CLIP vision layers and post-LayerNorm.
        # Handle both wrapped and direct vision-model layouts.
        vision_core = getattr(self.image_encoder, "vision_model", self.image_encoder)
        vision_layers = vision_core.encoder.layers
        for layer in vision_layers[-CFG["unfreeze_vision_layers"]:]:
            for p in layer.parameters(): p.requires_grad = True
        for p in vision_core.post_layernorm.parameters():
            p.requires_grad = True

        text_dim = self.text_encoder.config.hidden_size
        image_dim = self.image_encoder.config.hidden_size
        self.text_heads = ModalityHeads(text_dim, CFG["dropout"])
        self.image_heads = ModalityHeads(image_dim, CFG["dropout"])

        self.fusions = nn.ModuleDict({t: AdaptiveLateFusion(1) for t in CFG["binary_tasks"]})
        self.fusions["language"] = AdaptiveLateFusion(1)
        self.fusions["target"] = AdaptiveLateFusion(len(CFG["targets"]))

        # Model-parallel placement across two T4 GPUs.
        self.text_encoder.to(self.device_text)
        self.text_heads.to(self.device_text)
        self.fusions.to(self.device_text)
        self.image_encoder.to(self.device_image)
        self.image_heads.to(self.device_image)

    def forward(self, input_ids, attention_mask, pixel_values):
        input_ids = input_ids.to(self.device_text, non_blocking=True)
        attention_mask = attention_mask.to(self.device_text, non_blocking=True)
        pixel_values = pixel_values.to(self.device_image, non_blocking=True)

        text_h = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:,0]
        image_h = self.image_encoder(pixel_values=pixel_values).pooler_output

        text_logits = self.text_heads(text_h)
        image_logits_local = self.image_heads(image_h)
        image_logits = {k:v.to(self.device_text) for k,v in image_logits_local.items()}

        fused = {k:self.fusions[k](image_logits[k], text_logits[k]) for k in text_logits}
        return {"fused": fused, "text": text_logits, "image": image_logits}

    def fusion_report(self):
        report = {}
        for name, mod in self.fusions.items():
            w = torch.softmax(mod.weight_logits.detach().cpu(), dim=-1).numpy()
            tau = (F.softplus(mod.raw_temperature.detach().cpu()) + 1e-4).numpy()
            report[name] = {"weights_image_text": w.tolist(), "temperature": tau.tolist()}
        return report


In [ ]:

# Focal losses and training-split class weights.
class WeightedFocalCE(nn.Module):
    def __init__(self, class_weights, gamma=2.0, alpha=0.75):
        super().__init__()
        self.register_buffer("class_weights", torch.tensor(class_weights, dtype=torch.float32))
        self.gamma = float(gamma)
        self.alpha = float(alpha)
    def forward(self, logits, targets):
        w = self.class_weights.to(logits.device)
        ce = F.cross_entropy(logits, targets, weight=w, reduction="none")
        with torch.no_grad():
            pt = torch.softmax(logits, dim=-1).gather(1, targets.view(-1,1)).squeeze(1).clamp(1e-6, 1-1e-6)
        return (self.alpha * (1.0 - pt).pow(self.gamma) * ce).mean()

def inverse_frequency_weights(y, n_classes, power=1.0, max_ratio=None):
    y = np.asarray(y, dtype=np.int64)
    counts = np.bincount(y, minlength=n_classes).astype(np.float64)
    if np.any(counts == 0):
        raise ValueError(f"Class absent from train split: counts={counts}")
    w = (1.0 / counts) ** power
    if max_ratio is not None:
        lo = w.min()
        w = np.minimum(w, lo * max_ratio)
    w = w / w.mean()
    return w.astype(np.float32)

def build_criteria(train_frame):
    crit = {}
    for task in CFG["binary_tasks"]:
        w = inverse_frequency_weights(train_frame[task], 2, CFG["binary_weight_power"], 10.0)
        crit[task] = WeightedFocalCE(w, CFG["focal_gamma"], CFG["focal_alpha"])
        print(task, "weights", w.tolist())
    w = inverse_frequency_weights(train_frame.language_id, len(CFG["languages"]), CFG["language_weight_power"], 8.0)
    crit["language"] = WeightedFocalCE(w, CFG["focal_gamma"], CFG["focal_alpha"])
    print("language weights", w.tolist())
    target_crit = []
    for t in CFG["targets"]:
        w = inverse_frequency_weights(
            train_frame[f"target__{t}"], 2,
            CFG["target_weight_power"], CFG["target_weight_ratio_cap"]
        )
        target_crit.append(WeightedFocalCE(w, CFG["focal_gamma"], CFG["focal_alpha"]))
        print("target", t, "weights", w.tolist())
    crit["target"] = nn.ModuleList(target_crit)
    return crit

criteria = build_criteria(train_df)

def modality_task_loss(logits, labels, criteria):
    losses = []
    for task in CFG["binary_tasks"]:
        losses.append(criteria[task](logits[task], labels[task]))
    losses.append(criteria["language"](logits["language"], labels["language"]))
    target_losses = [criteria["target"][j](logits["target"][:,j,:], labels["target"][:,j]) for j in range(len(CFG["targets"]))]
    losses.append(torch.stack(target_losses).mean())
    return torch.stack(losses).mean()


In [ ]:
# Evaluation metrics and validation-based threshold calibration.

THRESHOLD_GRID = np.arange(
    CFG["threshold_min"],
    CFG["threshold_max"] + 1e-12,
    CFG["threshold_step"],
)

def logits_to_probs(logits):
    probs = {}
    for task in CFG["binary_tasks"]:
        probs[task] = torch.softmax(logits[task], dim=-1)[:, 1].numpy()
    probs["language"] = torch.softmax(logits["language"], dim=-1).numpy()
    probs["target"] = torch.softmax(logits["target"], dim=-1)[:, :, 1].numpy()
    return probs

def raw_thresholds():
    return {
        **{task: 0.5 for task in CFG["binary_tasks"]},
        "target": {t: 0.5 for t in CFG["targets"]},
    }

def metric_report(y_true, probs, thresholds):
    report = {}
    scalar_accs = []
    scalar_f1s = []

    for task in CFG["binary_tasks"]:
        pred = (probs[task] >= thresholds[task]).astype(int)
        acc = float(accuracy_score(y_true[task], pred))
        mf1 = float(f1_score(y_true[task], pred, average="macro", zero_division=0))
        report[task] = {"accuracy": acc, "macro_f1": mf1}
        scalar_accs.append(acc)
        scalar_f1s.append(mf1)

    lang_pred = probs["language"].argmax(axis=1)
    lang_acc = float(accuracy_score(y_true["language"], lang_pred))
    lang_mf1 = float(f1_score(y_true["language"], lang_pred, average="macro", zero_division=0))
    report["language"] = {"accuracy": lang_acc, "macro_f1": lang_mf1}
    scalar_accs.append(lang_acc)
    scalar_f1s.append(lang_mf1)

    target_pred = np.zeros_like(y_true["target"], dtype=int)
    per_target = {}
    for j, t in enumerate(CFG["targets"]):
        target_pred[:, j] = (probs["target"][:, j] >= thresholds["target"][t]).astype(int)
        per_target[t] = float(
            f1_score(y_true["target"][:, j], target_pred[:, j], average="binary", zero_division=0)
        )

    target_exact = float(np.mean(np.all(target_pred == y_true["target"], axis=1)))
    target_macro = float(f1_score(y_true["target"], target_pred, average="macro", zero_division=0))
    target_micro = float(f1_score(y_true["target"], target_pred, average="micro", zero_division=0))
    report["target"] = {
        "accuracy": target_exact,
        "exact_set_accuracy": target_exact,
        "macro_f1": target_macro,
        "micro_f1": target_micro,
        "per_label_f1": per_target,
    }

    five_acc = float(np.mean(scalar_accs))
    five_mf1 = float(np.mean(scalar_f1s))
    six_acc = float(np.mean(scalar_accs + [target_exact]))
    six_mf1 = float(np.mean(scalar_f1s + [target_macro]))

    report["aggregate_5task"] = {"accuracy": five_acc, "macro_f1": five_mf1}
    report["aggregate_6task"] = {"accuracy": six_acc, "macro_f1": six_mf1}
    report["overall_six_task_macro_f1"] = six_mf1  # Backward-compatible aggregate key.
    return report, target_pred

def selection_tuple(report, epoch):
    # Validation checkpoint ranking criterion:
    # maximize (5-task MF1, 6-task MF1, -|6-task gap|, 6-task accuracy, -step)
    five_mf1 = report["aggregate_5task"]["macro_f1"]
    six_mf1 = report["aggregate_6task"]["macro_f1"]
    six_acc = report["aggregate_6task"]["accuracy"]
    gap_pp = abs(six_acc - six_mf1) * 100.0
    return (five_mf1, six_mf1, -gap_pp, six_acc, -epoch)

def best_calibrated_threshold(y, p, default=0.5):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    n_pos = int(y.sum())
    n_neg = int(len(y) - n_pos)

    # Keep the default threshold for labels with insufficient validation support.
    if min(n_pos, n_neg) < CFG["threshold_min_support"]:
        return float(default), {
            "changed": False, "reason": "support_below_minimum",
            "positive_support": n_pos, "negative_support": n_neg,
        }

    base_pred = (p >= default).astype(int)
    base_f1 = float(f1_score(y, base_pred, average="macro", zero_division=0))
    best_t, best_f1 = float(default), base_f1

    for t in THRESHOLD_GRID:
        pred = (p >= t).astype(int)
        score = float(f1_score(y, pred, average="macro", zero_division=0))
        if score > best_f1 + 1e-12:
            best_t, best_f1 = float(t), score

    improvement = best_f1 - base_f1
    if improvement < CFG["threshold_min_improvement"]:
        return float(default), {
            "changed": False, "reason": "improvement_below_minimum",
            "base_macro_f1": base_f1, "best_macro_f1": best_f1,
            "improvement": improvement,
        }

    return best_t, {
        "changed": True, "reason": "accepted",
        "base_macro_f1": base_f1, "best_macro_f1": best_f1,
        "improvement": improvement,
    }

def calibrate_thresholds_after_selection(y_true, probs):
    th = raw_thresholds()
    audit = {}

    for task in CFG["binary_tasks"]:
        th[task], audit[task] = best_calibrated_threshold(y_true[task], probs[task], 0.5)

    audit["target"] = {}
    for j, t in enumerate(CFG["targets"]):
        th["target"][t], audit["target"][t] = best_calibrated_threshold(
            y_true["target"][:, j], probs["target"][:, j], 0.5
        )
    return th, audit

def frame_truth(frame):
    y = {t: frame[t].to_numpy(dtype=int) for t in CFG["binary_tasks"]}
    y["language"] = frame.language_id.to_numpy(dtype=int)
    y["target"] = frame[[f"target__{t}" for t in CFG["targets"]]].to_numpy(dtype=int)
    return y

y_val = frame_truth(val_df)
# Locked-test labels are excluded from development-time model selection and calibration.


In [ ]:
# FacetGate data loaders.
# Construct the test DataLoader only after all five development states are frozen.

val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"] * 4,
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True,
)

@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    all_logits = {k: [] for k in [*CFG["binary_tasks"], "language", "target"]}
    ids = []
    for batch in tqdm(loader, leave=False):
        out = model(
            batch["input_ids"],
            batch["attention_mask"],
            batch["pixel_values"],
        )["fused"]
        for k, v in out.items():
            all_logits[k].append(v.detach().float().cpu())
        ids.extend(list(batch["image"]))
    return ids, {k: torch.cat(v, dim=0) for k, v in all_logits.items()}

def make_exact_epoch_loader(epoch, seed):
    # Use each of the 7,454 training examples once per epoch.
    rng = np.random.default_rng(int(seed) + int(epoch))
    order = np.arange(len(train_ds), dtype=np.int64)
    rng.shuffle(order)

    if len(order) != 7454 or len(np.unique(order)) != 7454:
        raise RuntimeError("TRAIN order must contain exactly 7,454 unique rows.")

    subset = torch.utils.data.Subset(train_ds, order.tolist())
    loader = DataLoader(
        subset,
        batch_size=CFG["batch_size"],   # Four examples per microbatch.
        shuffle=False,
        drop_last=False,
        num_workers=CFG["num_workers"],
        pin_memory=True,
        persistent_workers=CFG["num_workers"] > 0,
    )
    return loader, order

print("Validation loader ready.")
print("Locked TEST loader: NOT CONSTRUCTED.")

In [ ]:
# Five-seed FacetGate orchestration.
COMMON_PROTOCOL = json.loads(r"""{"schema":"M3Offend_SharedForward_DualCheckpoint_Gated_SHTE_Common_Protocol_v1_0_1","status":"prospective_frozen_before_execution_final_corrected","protocol_name":"Shared-Forward Dual-Checkpoint Gated-SHTE","protocol_version":"1.0.1","scientific_non_equivalence":{"existing_facetgate_v1_0_sha256":"4107b18af7a9ad112449666bd97a037d8807b3c8a583a405130d9daa151e0af1","parent_frozen_common_gated_v1_1_sha256":"627bdf5a77ab694581ff590c32f8885c901ebad09e8df86dbab07762caaf95dd","rule":"Do not relabel previously executed frozen-parent FacetGate results as simultaneous Gated-SHTE results."},"dataset":{"release":"M3Offend","rows":9318,"split_counts":{"train":7454,"validation":932,"locked_test":932},"labels_sha256":"9cb9afdd88c9a750d3eb6f36aa512dc6ad5b3067aedfac79b138b7fef64a9bb6","ocr_sha256":"eed35bf77f749e98d9222fd571d149b99c4c6a610e29bc5db5783d3489492c0c","split_manifest_sha256":"bbcf7e13fc0aab4ac3b5f58a2ebf8880740ed60ad7ecb2cb61910b0497d1b4f0","canonical_row_sha256":"440f8e6982cf4272fc0b6c3e3e6fc9b079a02ce01cd5350a02be0d4a9fa1cf39","ordered_test_identity_sha256":"5f6ac881e9026d682f2f41d65ed1f08e1861d8c8f030bae04d98f9b9db6419db","image_bundle_sha256":"9e134f26c63c23ba871c954ee1d7e03499a47ac3f9bd49d12e0c5ffa977665af","max_ocr_tokens":384,"task_instruction":"Encode this meme for six independent annotations: offensive, vulgar, abuse, sarcastic, language (Bangla/English/Banglish), and zero or more of seven real target groups. Do not impose logical implications between tasks. "},"tasks":{"binary":["offensive","vulgar","abuse","sarcastic"],"language":["Bangla","English","Banglish"],"target":["Individual","Gender","Political","Social","National","Religious","Others"],"target_none_encoding":"all-seven-target-logits negative / empty real-target set"},"publication_seeds":[42,1337,2026,31415,271828],"single_pass_invariant":{"one_native_backbone_forward_per_microbatch":true,"same_visual_text_representation_fed_to_both_branches":true,"gated_branch":"original Frozen Common Gated v1.1 six-task path retained as anchor/generalist","shte_branch":"FacetGate v1.0 four-view Target specialist","shte_inputs_detached_from_gated_graph":true,"shte_gradient_must_not_update_gated_or_backbone":true,"separate_rng_stream_for_shte":true,"purpose":"train both branches on the same sample sequence without a second image/backbone pass and without allowing SHTE gradients to perturb the original Frozen Common Gated v1.1 scientific trajectory"},"gated_branch_common":{"quantization":"4-bit NF4","double_quantization":true,"compute_dtype":"float16","fusion_hidden_size":1024,"shared_hidden_size":1024,"task_adapter_hidden_size":256,"dropout":0.1,"modality_dropout_probability":0.1,"lora_r":16,"lora_alpha":32,"lora_dropout":0.05,"vision_lora_enabled":false,"language_lora_enabled":true,"projector_or_merger_lora_enabled":true,"learning_rate_lora":0.0002,"learning_rate_heads":0.0005,"optimizer":"AdamW","scheduler":"cosine_with_warmup","weight_decay":0.01,"warmup_ratio":0.05,"max_grad_norm":1.0,"loss_family":"A1_POS_WEIGHTED_BCE","positive_weight_cap":8.0,"auxiliary_lm_loss_weight":0.0,"class_weight_source":"training split only","task_weights":{"offensive":1.0,"vulgar":1.0,"abuse":1.0,"sarcastic":1.0,"language":1.0,"target":0.5}},"shte_target_branch":{"common_interface":["v","t","J","zG_target"],"dim":1024,"views":["v","t","LayerNorm(v*t)","LayerNorm(abs(v-t))"],"view_gate":"Linear(4096,128)->GELU->Linear(128,4)->softmax","expert_shared":"Linear(1024,1024)->GELU->Dropout(0.10)","expert_adapter":"LayerNorm(1024)->Linear(1024,256)->GELU->Dropout(0.10)->Linear(256,1024)->Dropout(0.10)+residual","expert_head":"Linear(1024,7)","selector_features":["pG","pH","abs(pH-pG)","abs(pG-0.5)","abs(pH-0.5)","1-cos(v,t)"],"selector":"shared MLP 6->16->GELU->1->sigmoid, applied label-wise","final_logits":"zStar = zG + selector*(zH-zG)","loss":"L_target + 0.05*L_selector + 0.0001*L_change","selector_teacher":"q=sigmoid((BCE(zG,y)-BCE(zH,y))/0.10), detached","positive_weight_cap":8.0,"learning_rate":0.0005,"weight_decay":0.01,"optimizer":"AdamW","scheduler":"cosine_with_warmup","warmup_ratio":0.05,"max_grad_norm":1.0,"dropout":0.1},"training":{"epochs":2,"world_size":2,"per_device_batch_size":1,"gradient_accumulation_per_rank":8,"nominal_effective_global_batch":16,"optimizer_steps_per_epoch":466,"optimizer_steps_total":932,"candidate_steps":[200,400,600,800,932],"same_sample_order_for_both_branches":true,"no_duplicate_real_example_contribution":true,"partial_final_group_real_examples":14,"partial_group_normalization":"exact mean over 14 real examples"},"checkpoint_selection":{"gated_native_rule":"(5task MF1, 6task MF1, -abs(6task gap pp), 6task accuracy, -step) on validation raw@0.5","gated_selected_state_includes":["backbone/LoRA","Gated projections/fusion/shared/adapters/heads","optimizer-independent inference state"],"target_candidate_states_saved_at":[200,400,600,800,932],"compatibility_replay":{"required":true,"procedure":"After selecting the Gated checkpoint c*, freeze c*, run validation backbone once to cache v,t,J,zG_target, then replay every saved SHTE candidate on that fixed cache.","reason":"SHTE checkpoints trained against evolving Gated representations cannot be safely combined with c* without validating the actual composite."},"target_rule_after_compatibility_replay":["Target Macro-F1 raw@0.5 maximize","Target mAP/AUPRC maximize","Target exact-set accuracy raw@0.5 maximize","earlier SHTE step minimize"],"independent_final_states":true},"calibration":{"source_split":"validation only","grid":{"start":0.05,"end":0.95,"step":0.025},"min_support":5,"min_f1_improvement_over_0p5":0.005,"target_tie_break":["closest threshold to 0.5","lower threshold"],"core5":"apply original Frozen Common Gated v1.1 calibration to selected Gated checkpoint","target":"calibrate seven Target thresholds only after compatible SHTE checkpoint is selected"},"test_policy":{"locked_test_forwards_during_development":0,"test_must_not_select_checkpoint":true,"test_must_not_select_threshold":true,"test_must_not_select_numerical_repair":true},"execution_modes":{"SIMULTANEOUS_FROM_SCRATCH":{"paper_status":"new protocol result","description":"Gated and SHTE train on the same sample stream in one outer loop and one backbone forward per microbatch; SHTE uses detached Gated interface and a separate optimizer/RNG stream.","trigger":"automatically selected only when no valid same-seed Gated parent is supplied/found"},"REUSE_EXISTING_GATED":{"paper_status":"existing frozen-parent FacetGate reproduction path, not equivalent to simultaneous result","description":"Load exact same-seed selected Gated checkpoint and train only SHTE/FacetGate Target specialist.","non_mixing_rule":"Do not aggregate or label this mode as simultaneous-from-scratch.","trigger":"automatically selected when exactly one valid same-seed Gated parent is supplied/found"}},"promotion":{"development_first_seed":42,"minimum_requirements":["Final composite Core-5 logits/predictions must equal the selected Gated checkpoint c* Core-5 outputs within absolute tolerance 1e-10","For SIMULTANEOUS_FROM_SCRATCH, validation 5-task Macro-F1 after original Gated calibration must not drop by more than 0.5 percentage point versus the matched same-backbone + same-seed standalone Frozen Common Gated v1.1 reference","Target Macro-F1 must improve by at least +1.0 percentage point","Target mAP must not decrease","rare-target Macro-F1 must not decrease by more than 0.5 percentage point","paired item and group bootstrap deltas should be reported"],"multi_backbone_rule":"freeze the rule before observing new model results; any redesign after seeing outcomes requires protocol v1.1+"},"artifact_requirements":["canonical protocol JSON and SHA-256","model-native binding JSON and SHA-256","seed and sample-order manifest","Gated candidate checkpoints and validation metrics","SHTE candidate checkpoints and native-pair validation metrics","selected Gated checkpoint c*","fixed-c* validation feature cache manifest","SHTE compatibility-replay table across all candidate states","selected SHTE checkpoint t*","Core-5 thresholds and seven Target thresholds","TEST forward counter","final composite binding manifest with hashes"],"automatic_startup_orchestration":{"enabled":true,"decision_point":"before any training or backbone forward","gated_input_search_scope":["explicit GATED_SOURCE_OVERRIDE if supplied","mounted Kaggle/input or equivalent anonymous-repository artifact roots"],"valid_existing_gated_requirements":["exact same backbone model ID","exact frozen model revision","exact RUN_SEED","validation-selected Gated checkpoint provenance","classifier heads/state present","adapter/LoRA state and config present","frozen Gated validation thresholds present","dataset/split binding compatible with M3Offend","parent Frozen Common Gated v1.1 protocol binding","development TEST-forward count equals zero"],"decision_rule":{"exactly_one_valid_same_seed_gated_input":{"mode":"REUSE_EXISTING_GATED","action":"load and freeze the supplied validation-selected Gated checkpoint; skip Gated training; train only the SHTE/FacetGate Target specialist"},"no_valid_gated_input_and_no_conflicting_intended_parent":{"mode":"SIMULTANEOUS_FROM_SCRATCH","action":"initialize the original Gated branch and SHTE Target specialist together; for each micro-batch perform one native VLM forward and update both branches in the same outer training loop with independent optimizers/RNG streams"},"mismatched_incomplete_or_ambiguous_gated_input":{"mode":"FAIL_CLOSED","action":"stop before training and report the rejected provenance; do not silently substitute a different seed/model/checkpoint"}},"simultaneous_from_scratch_per_microbatch":["read one image + cleaned OCR sample/micro-batch","perform exactly one native VLM/backbone forward","compute Gated v,t,J and six-task logits/loss using the original Gated path","detach v,t,J,zG_target for the SHTE branch","compute SHTE Target logits/loss on the same sample representation","backpropagate/update Gated with its optimizer","backpropagate/update SHTE with its separate optimizer","advance to the next sample/micro-batch"],"independent_checkpoint_tracking":{"candidate_steps":[200,400,600,800,932],"gated":"save/rank Gated candidates by the original Frozen Common Gated validation rule","shte":"save every SHTE candidate; final SHTE selection occurs after fixed-selected-Gated compatibility replay"},"provenance_log_required":true,"required_log_fields":["RUN_SEED","backbone ID and revision","mode selected","all discovered Gated candidates and rejection reasons","selected/reused Gated checkpoint hash","selected SHTE checkpoint hash","TEST forward counter"]},"core5_preservation":{"definition":["offensive","vulgar","abuse","sarcastic","language"],"gated_protocol_anchor_sha256":"627bdf5a77ab694581ff590c32f8885c901ebad09e8df86dbab07762caaf95dd","shte_must_not_modify_core5_logits_or_predictions":true,"composite_core5_identity_reference":"the selected Gated checkpoint c* used in the final composite","composite_core5_identity_absolute_tolerance":1e-10,"simultaneous_mode_reference":"matched same-backbone + same-seed standalone Frozen Common Gated v1.1 result","no_regression_metric":"validation 5-task Macro-F1 after applying the original Frozen Common Gated v1.1 validation calibration","maximum_allowed_drop_percentage_points":0.5,"decision_rule":"PASS only if final composite Core-5 outputs are identical to the selected Gated c* outputs within absolute tolerance 1e-10 and the simultaneous-mode 5-task Macro-F1 drop versus the matched same-backbone/same-seed Gated reference is <= 0.5 percentage point."},"revision_history":{"supersedes_draft_identifier":"4c64f61f881831408b953d213351708b59ba022696f0ec0d0d18c39ee9a10846","reason":["clarified that simultaneous Gated training is not frozen, but is isolated from SHTE gradients","froze the previously unspecified Core-5 no-regression tolerance at 0.5 percentage point on validation 5-task Macro-F1","made final-composite Core-5 identity explicit at absolute tolerance 1e-10","defined a reproducible canonical JSON hashing procedure"],"scientific_architecture_changed":false,"execution_modes_changed":false,"checkpoint_selection_changed":false,"test_policy_changed":false},"integrity":{"hash_algorithm":"SHA-256","canonicalization":"UTF-8 JSON; sort_keys=True; separators=(',', ':'); ensure_ascii=False; exclude the field integrity.protocol_canonical_sha256 itself before hashing","protocol_canonical_sha256":"37b7d30a994fb98225d49d76cca5768a838d456abf844dc1a5d93019a0cd15aa"}}""")
EXPECTED_PROTOCOL_SHA256 = "37b7d30a994fb98225d49d76cca5768a838d456abf844dc1a5d93019a0cd15aa"

def canonical_protocol_sha(obj):
    x = json.loads(json.dumps(obj))
    x["integrity"].pop("protocol_canonical_sha256", None)
    return hashlib.sha256(
        json.dumps(x, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    ).hexdigest()

if canonical_protocol_sha(COMMON_PROTOCOL) != EXPECTED_PROTOCOL_SHA256:
    raise RuntimeError("Protocol SHA mismatch.")

PUBLICATION_SEEDS = [42, 1337, 2026, 31415, 271828]
ALL5_ROOT = Path("/kaggle/working/M3Offend_9318_BanglaBERT_CLIP_FacetGate_ALL5_FINAL_LOCKED_TEST")
ALL5_ROOT.mkdir(parents=True, exist_ok=True)

TEST_FORWARD_COUNT_DURING_DEVELOPMENT = 0
LOCKED_TEST_SEED_PASSES = 0
DEVELOPMENT_PHASE_FROZEN = False

PARENT_MANIFEST_NAME = "FACETGATE_GATED_PARENT_MANIFEST.json"

FACET_BINDING = {
    "schema": "M3Offend_BanglaBERT_CLIP_FacetGate_Binding_v1_0_1_ALL5",
    "experiment_name": "FacetGate",
    "common_protocol_sha256": EXPECTED_PROTOCOL_SHA256,
    "text_model": CFG["text_model"],
    "vision_model": CFG["vision_model"],
    "text_revision": CFG["text_revision"],
    "vision_revision": CFG["vision_revision"],
    "parent_model_source_sha256": "49aca4ff7a3825aabc18a8079d09bb6b981786406424915072ade382c8a870b0",
    "controlled_binding": True,
    "locked_test_policy": "All five development states frozen before first TEST model forward.",
}
FACET_BINDING_SHA256 = hashlib.sha256(
    json.dumps(FACET_BINDING, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
).hexdigest()

(ALL5_ROOT / "COMMON_PROTOCOL_v1_0_1.json").write_text(
    json.dumps(COMMON_PROTOCOL, ensure_ascii=False, indent=2), encoding="utf-8"
)
(ALL5_ROOT / "BANGLABERT_FACETGATE_BINDING.json").write_text(
    json.dumps(FACET_BINDING, ensure_ascii=False, indent=2), encoding="utf-8"
)

def _safe_json(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return None

def seed_paths(seed):
    root = ALL5_ROOT / f"seed_{int(seed)}"
    facet = root / "facetgate"
    gated_candidates = facet / "gated_candidates"
    target_candidates = facet / "target_candidates"
    for p in [root, facet, gated_candidates, target_candidates]:
        p.mkdir(parents=True, exist_ok=True)
    return {
        "root": root,
        "facet": facet,
        "gated_candidates": gated_candidates,
        "target_candidates": target_candidates,
    }

def discover_parent_for_seed(seed):
    # Search only mounted input artifacts for reusable parent checkpoints.
    
    valid = []
    rejected = []
    same_seed_looking = []

    for mpath in sorted(Path("/kaggle/input").rglob(PARENT_MANIFEST_NAME)):
        obj = _safe_json(mpath)
        if not isinstance(obj, dict):
            continue
        if int(obj.get("run_seed", -1)) != int(seed):
            continue

        same_seed_looking.append(mpath)
        reasons = []

        required_equal = {
            "run_seed": int(seed),
            "text_model": CFG["text_model"],
            "vision_model": CFG["vision_model"],
            "text_revision": CFG["text_revision"],
            "vision_revision": CFG["vision_revision"],
            "dataset_sha256": CFG["expected_sha256"],
            "parent_model_source_sha256": FACET_BINDING["parent_model_source_sha256"],
            "protocol_sha256": EXPECTED_PROTOCOL_SHA256,
        }
        for k, expected in required_equal.items():
            if obj.get(k) != expected:
                reasons.append(f"{k} mismatch")

        if obj.get("validation_selected") is not True:
            reasons.append("not validation-selected")
        if int(obj.get("test_forward_count", -1)) != 0:
            reasons.append("development TEST-forward count is not zero")

        checkpoint = mpath.parent / str(obj.get("selected_checkpoint", ""))
        thresholds = mpath.parent / str(obj.get("thresholds_file", ""))
        if not checkpoint.is_file():
            reasons.append("selected checkpoint missing")
        if not thresholds.is_file():
            reasons.append("threshold file missing")

        if reasons:
            rejected.append({"manifest": str(mpath), "reasons": reasons})
        else:
            valid.append({
                "manifest_path": mpath,
                "manifest": obj,
                "checkpoint": checkpoint,
                "thresholds": thresholds,
            })

    if len(valid) > 1:
        raise RuntimeError(
            f"FAIL_CLOSED seed {seed}: multiple valid same-seed Gated parents found."
        )
    if len(valid) == 1:
        return "REUSE_EXISTING_GATED", valid[0], rejected
    if same_seed_looking:
        raise RuntimeError(
            f"FAIL_CLOSED seed {seed}: parent-looking artifact exists but is invalid. "
            + json.dumps(rejected, indent=2)
        )
    return "SIMULTANEOUS_FROM_SCRATCH", None, rejected

print("All-five FacetGate orchestration ready.")
print("Seeds:", PUBLICATION_SEEDS)
print("Development TEST forwards:", TEST_FORWARD_COUNT_DURING_DEVELOPMENT)

In [ ]:
# Deterministic validation pHash groups for paired group bootstrap.
from scipy.fftpack import dct

VALIDATION_GROUPING_VERSION = "validation-standard-phash64-hamming6-connected-components-v1"
VALIDATION_GROUPING_SPEC = (
    "VALIDATION only; grayscale; resize 32x32 Lanczos; scipy.fftpack DCT-II rows+cols; "
    "top-left 8x8; bits=(coefficient>median); 64-bit row-major pHash; "
    "edge when Hamming distance<=6; groups=connected components; group ID=min official_row_index."
)
EXPECTED_GROUPING_SPEC_SHA = "07957a24191a29633e6386f0388ca990cd38e9e301061c92fd5a38990c0ed47f"
observed_group_spec_sha = hashlib.sha256(
    VALIDATION_GROUPING_SPEC.encode("utf-8")
).hexdigest()
if observed_group_spec_sha != EXPECTED_GROUPING_SPEC_SHA:
    raise RuntimeError("Validation grouping specification drift.")

def _phash64(path):
    with Image.open(path) as im:
        im = im.convert("L").resize((32, 32), Image.Resampling.LANCZOS)
        px = np.asarray(im, dtype=np.float64)
    coeff = dct(dct(px, axis=0), axis=1)
    low = coeff[:8, :8]
    med = float(np.median(low))
    bits = (low > med).reshape(-1)
    h = 0
    for bit in bits:
        h = (h << 1) | int(bool(bit))
    return int(h)

val_group_df = val_df.copy()
if "official_row_index" not in val_group_df.columns:
    raise RuntimeError("official_row_index missing; cannot reproduce FacetGate group bootstrap.")

val_indices = val_group_df["official_row_index"].astype(int).tolist()
val_hashes = []
for n, image in enumerate(val_group_df["image"].astype(str).tolist(), 1):
    val_hashes.append(_phash64(IMAGE_DIR / image))
    if n % 200 == 0 or n == len(val_group_df):
        print(f"[VAL PHASH] {n}/{len(val_group_df)}")

parent_uf = list(range(len(val_hashes)))
rank_uf = [0] * len(val_hashes)

def _uf_find(a):
    while parent_uf[a] != a:
        parent_uf[a] = parent_uf[parent_uf[a]]
        a = parent_uf[a]
    return a

def _uf_union(a, b):
    ra, rb = _uf_find(a), _uf_find(b)
    if ra == rb:
        return
    if rank_uf[ra] < rank_uf[rb]:
        ra, rb = rb, ra
    parent_uf[rb] = ra
    if rank_uf[ra] == rank_uf[rb]:
        rank_uf[ra] += 1

for i in range(len(val_hashes)):
    hi = val_hashes[i]
    for j in range(i + 1, len(val_hashes)):
        if (hi ^ val_hashes[j]).bit_count() <= 6:
            _uf_union(i, j)

components = {}
for i, official_idx in enumerate(val_indices):
    components.setdefault(_uf_find(i), []).append(int(official_idx))

group_by_index = {}
for members in components.values():
    gid = f"ph6_{min(members):06d}"
    for official_idx in members:
        group_by_index[int(official_idx)] = gid

VAL_BOOTSTRAP_GROUPS = np.asarray(
    [group_by_index[int(x)] for x in val_indices],
    dtype=str,
)
if len(VAL_BOOTSTRAP_GROUPS) != 932:
    raise RuntimeError("Validation grouping row-count drift.")
if len(np.unique(VAL_BOOTSTRAP_GROUPS)) != 922:
    raise RuntimeError(
        f"Validation grouping identity drift: expected 922 groups, "
        f"got {len(np.unique(VAL_BOOTSTRAP_GROUPS))}."
    )

group_payload = "\n".join(
    f"{int(idx)}\t{grp}"
    for idx, grp in zip(val_indices, VAL_BOOTSTRAP_GROUPS)
) + "\n"

group_manifest = {
    "version": VALIDATION_GROUPING_VERSION,
    "spec_sha256": observed_group_spec_sha,
    "rows": 932,
    "unique_groups": int(len(np.unique(VAL_BOOTSTRAP_GROUPS))),
    "mapping_sha256": hashlib.sha256(group_payload.encode("utf-8")).hexdigest(),
}
(ALL5_ROOT / "validation_group_manifest.json").write_text(
    json.dumps(group_manifest, indent=2),
    encoding="utf-8",
)
print("[VALIDATION GROUP PASS]", group_manifest)

In [ ]:
# FacetGate Target specialist for the BanglaBERT+CLIP binding.

FACET_CFG = {
    "epochs": 2,
    "micro_batch_size": CFG["batch_size"],  # microbatch size
    "gradient_accumulation_microbatches": 4,
    "nominal_effective_batch": 16,
    "steps_per_epoch": 466,
    "total_steps": 932,
    "candidate_steps": [200, 400, 600, 800, 932],
    "dim": 1024,
    "adapter_hidden": 256,
    "dropout": 0.10,
    "lr": 5e-4,
    "weight_decay": 0.01,
    "warmup_ratio": 0.05,
    "max_grad_norm": 1.0,
    "positive_weight_cap": 8.0,
    "selector_teacher_temperature": 0.10,
    "selector_loss_weight": 0.05,
    "change_loss_weight": 0.0001,
    "core5_identity_tolerance": 1e-10,
    "simultaneous_max_core5_drop_pp": 0.5,
    "bootstrap_repeats": 10000,
}

class DedicatedDropout(nn.Module):
    """Dropout with a FacetGate-only RNG stream; never consumes parent/global RNG."""
    def __init__(self, p, seed, device="cuda:0"):
        super().__init__()
        self.p = float(p)
        self.seed = int(seed)
        self.preferred_device = str(device)
        self._generators = {}

    def _generator_for(self, x):
        # Maintain a dedicated generator for each tensor device.
        
        key = str(x.device)
        if key not in self._generators:
            g = torch.Generator(device=x.device)
            g.manual_seed(self.seed)
            self._generators[key] = g
        return self._generators[key]

    def forward(self, x):
        if not self.training or self.p <= 0:
            return x
        keep = torch.rand(
            x.shape,
            generator=self._generator_for(x),
            device=x.device,
            dtype=torch.float32,
        ) >= self.p
        return x * keep.to(x.dtype) / (1.0 - self.p)

class ResidualFacetAdapter(nn.Module):
    def __init__(self, hidden=1024, bottleneck=256, dropout=0.10, seed=1):
        super().__init__()
        self.norm = nn.LayerNorm(hidden)
        self.fc1 = nn.Linear(hidden, bottleneck)
        self.fc2 = nn.Linear(bottleneck, hidden)
        self.drop1 = DedicatedDropout(dropout, seed + 1)
        self.drop2 = DedicatedDropout(dropout, seed + 2)

    def forward(self, x):
        z = self.fc1(self.norm(x))
        z = F.gelu(z)
        z = self.drop1(z)
        z = self.fc2(z)
        z = self.drop2(z)
        return x + z

class FacetGateTarget(nn.Module):
    def __init__(self, image_native_dim, text_native_dim, seed):
        super().__init__()
        D = FACET_CFG["dim"]

        # Interface projections are confined to the FacetGate specialist.
        
        self.visual_projection = nn.Sequential(
            nn.LayerNorm(image_native_dim),
            nn.Linear(image_native_dim, D),
            nn.GELU(),
        )
        self.text_projection = nn.Sequential(
            nn.LayerNorm(text_native_dim),
            nn.Linear(text_native_dim, D),
            nn.GELU(),
        )
        self.joint_norm = nn.LayerNorm(D)

        self.product_norm = nn.LayerNorm(D)
        self.difference_norm = nn.LayerNorm(D)
        self.target_view_gate = nn.Sequential(
            nn.Linear(4 * D, 128),
            nn.GELU(),
            nn.Linear(128, 4),
        )

        self.target_expert_shared = nn.Linear(D, D)
        self.shared_drop = DedicatedDropout(0.10, seed + 10)
        self.target_expert_adapter = ResidualFacetAdapter(
            D, 256, 0.10, seed + 20
        )
        self.target_expert_head = nn.Linear(D, 7)
        self.target_selector = nn.Sequential(
            nn.Linear(6, 16),
            nn.GELU(),
            nn.Linear(16, 1),
        )

    def forward(self, image_native, text_native, zG):
        v = self.visual_projection(image_native.float())
        t = self.text_projection(text_native.float())

        # J is retained for protocol accounting but is not used by the four-view specialist.
        
        J = self.joint_norm(0.5 * (v + t))

        prod = self.product_norm(v * t)
        diff = self.difference_norm(torch.abs(v - t))
        views = torch.stack([v, t, prod, diff], dim=1)

        gate_in = torch.cat([v, t, prod, diff], dim=-1)
        view_weights = F.softmax(
            self.target_view_gate(gate_in.float()),
            dim=-1,
        )
        h = (views.float() * view_weights.unsqueeze(-1)).sum(dim=1)

        h = F.gelu(self.target_expert_shared(h))
        h = self.shared_drop(h)
        zH = self.target_expert_head(
            self.target_expert_adapter(h)
        )

        pG = torch.sigmoid(zG.float())
        pH = torch.sigmoid(zH.float())
        d_it = (
            1.0 - F.cosine_similarity(v.float(), t.float(), dim=-1)
        ).clamp(0, 2).unsqueeze(-1).expand(-1, 7)

        selector_features = torch.stack(
            [
                pG,
                pH,
                torch.abs(pH - pG),
                torch.abs(pG - 0.5),
                torch.abs(pH - 0.5),
                d_it,
            ],
            dim=-1,
        )
        selector_logit = self.target_selector(
            selector_features.reshape(-1, 6)
        ).reshape(-1, 7)
        selector = torch.sigmoid(selector_logit)

        zStar = zG.float() + selector * (zH.float() - zG.float())

        return zStar, {
            "v": v,
            "t": t,
            "J": J,
            "zH": zH,
            "selector_logit": selector_logit,
            "selector": selector,
            "view_weights": view_weights,
            "image_text_disagreement": d_it[:, 0],
        }

def target_positive_weights(frame):
    vals = []
    for lab in CFG["targets"]:
        y = frame[f"target__{lab}"].to_numpy(dtype=int)
        pos = int(y.sum())
        neg = int(len(y) - pos)
        if pos <= 0:
            raise RuntimeError(f"No TRAIN positives for Target={lab}")
        vals.append(min(neg / pos, FACET_CFG["positive_weight_cap"]))
    return torch.tensor(vals, dtype=torch.float32, device="cuda:0")

FACET_POS_WEIGHT = target_positive_weights(train_df)

def facet_loss_vector(zStar, extra, zG, y):
    # Mean loss across the seven Target labels for each example.
    L_target = F.binary_cross_entropy_with_logits(
        zStar.float(),
        y.float(),
        pos_weight=FACET_POS_WEIGHT,
        reduction="none",
    ).mean(dim=1)

    with torch.no_grad():
        g_loss = F.binary_cross_entropy_with_logits(
            zG.float(), y.float(), reduction="none"
        )
        h_loss = F.binary_cross_entropy_with_logits(
            extra["zH"].float(), y.float(), reduction="none"
        )
        q = torch.sigmoid(
            (g_loss - h_loss) / FACET_CFG["selector_teacher_temperature"]
        )

    L_selector = F.binary_cross_entropy_with_logits(
        extra["selector_logit"].float(),
        q.float(),
        reduction="none",
    ).mean(dim=1)

    L_change = (
        extra["selector"].float().pow(2)
        * (extra["zH"].float() - zG.float()).pow(2)
    ).mean(dim=1)

    return (
        L_target
        + FACET_CFG["selector_loss_weight"] * L_selector
        + FACET_CFG["change_loss_weight"] * L_change
    )

def target_metric_bundle(y, prob, thr):
    y = np.asarray(y, dtype=np.int64)
    p = np.asarray(prob, dtype=np.float64)
    th = np.asarray(thr, dtype=np.float64).reshape(1, -1)
    pred = (p >= th).astype(np.int64)

    per_f1 = [
        float(f1_score(y[:, j], pred[:, j], zero_division=0))
        for j in range(7)
    ]
    per_ap = []
    for j in range(7):
        if len(np.unique(y[:, j])) < 2:
            per_ap.append(float("nan"))
        else:
            per_ap.append(float(average_precision_score(y[:, j], p[:, j])))

    return {
        "macro_f1": float(np.mean(per_f1)),
        "map": float(np.nanmean(per_ap)),
        "micro_f1": float(
            f1_score(y, pred, average="micro", zero_division=0)
        ),
        "samples_f1": float(
            f1_score(y, pred, average="samples", zero_division=0)
        ),
        "exact_set_accuracy": float(np.mean(np.all(y == pred, axis=1))),
        "hamming_loss": float(hamming_loss(y, pred)),
        "per_label_f1": dict(zip(CFG["targets"], per_f1)),
        "per_label_ap": dict(zip(CFG["targets"], per_ap)),
        "rare_target_macro_f1": float(
            np.mean(
                [
                    per_f1[CFG["targets"].index("National")],
                    per_f1[CFG["targets"].index("Religious")],
                ]
            )
        ),
    }



# CPU-only shape check without dataset or model inference.
with torch.random.fork_rng(devices=[], enabled=True):
    torch.manual_seed(42)
    _smoke = FacetGateTarget(768, 768, 1_000_045).cpu()
    _z, _ex = _smoke(torch.randn(5,768), torch.randn(5,768), torch.randn(5,7))
    assert _z.shape == (5,7)
    assert _ex["v"].shape == (5,1024)
    assert _ex["t"].shape == (5,1024)
    assert _ex["J"].shape == (5,1024)
del _smoke, _z, _ex
print("[FacetGate specialist synthetic shape PASS]")

In [ ]:
# Five-seed FacetGate development engine.

_CAPTURE = {}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

def trainable_groups(model):
    image_params, text_params, fusion_params = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("image_encoder"):
            image_params.append(p)
        elif name.startswith("text_encoder"):
            text_params.append(p)
        else:
            fusion_params.append(p)
    return image_params, text_params, fusion_params

def trainable_state(model):
    return {
        name: p.detach().cpu().clone()
        for name, p in model.named_parameters()
        if p.requires_grad
    }

def load_trainable_state(model, state):
    named = dict(model.named_parameters())
    expected = {n for n,p in model.named_parameters() if p.requires_grad}
    observed = set(state)
    if expected != observed:
        raise RuntimeError(
            f"trainable-state mismatch missing={sorted(expected-observed)[:10]} "
            f"extra={sorted(observed-expected)[:10]}"
        )
    with torch.no_grad():
        for name,tensor in state.items():
            named[name].copy_(tensor.to(named[name].device, dtype=named[name].dtype))

def divide_gradients(params, divisor):
    divisor = float(divisor)
    for p in params:
        if p.grad is not None:
            p.grad.div_(divisor)

def _text_hook(module, inputs, output):
    _CAPTURE["text_native"] = output.last_hidden_state[:,0].detach()

def _image_hook(module, inputs, output):
    _CAPTURE["image_native"] = output.pooler_output.detach()

def parent_target_logodds(target_logits):
    return target_logits[:,:,1] - target_logits[:,:,0]

def gated_loss_scalar(outputs, labels):
    lf = modality_task_loss(outputs["fused"], labels, criteria)
    lt = modality_task_loss(outputs["text"], labels, criteria)
    li = modality_task_loss(outputs["image"], labels, criteria)
    return lf + CFG["aux_loss_weight"]*lt + CFG["aux_loss_weight"]*li

def make_facet_model(parent, seed):
    facet_seed = int(seed) + 1_000_003
    image_dim = int(parent.image_encoder.config.hidden_size)
    text_dim = int(parent.text_encoder.config.hidden_size)
    with torch.random.fork_rng(devices=[0], enabled=True):
        torch.manual_seed(facet_seed)
        torch.cuda.manual_seed_all(facet_seed)
        model = FacetGateTarget(image_dim, text_dim, facet_seed).to("cuda:0")
    return model

def make_parent_optimizer(parent):
    image_params, text_params, fusion_params = trainable_groups(parent)
    opt = torch.optim.AdamW(
        [
            {"params": image_params, "lr": CFG["image_lr"]},
            {"params": text_params, "lr": CFG["text_lr"]},
            {"params": fusion_params, "lr": CFG["fusion_lr"]},
        ],
        weight_decay=CFG["weight_decay"],
    )
    sched = get_cosine_schedule_with_warmup(
        opt,
        num_warmup_steps=int(FACET_CFG["total_steps"] * 0.05),
        num_training_steps=FACET_CFG["total_steps"],
    )
    return opt, sched, image_params, text_params, fusion_params

def make_facet_optimizer(facet):
    opt = torch.optim.AdamW(
        facet.parameters(),
        lr=FACET_CFG["lr"],
        weight_decay=FACET_CFG["weight_decay"],
    )
    sched = get_cosine_schedule_with_warmup(
        opt,
        num_warmup_steps=int(FACET_CFG["total_steps"] * FACET_CFG["warmup_ratio"]),
        num_training_steps=FACET_CFG["total_steps"],
    )
    return opt, sched

@torch.no_grad()
def evaluate_native_pair(parent, facet):
    # Preserve train/eval modes across candidate validation.
    
    parent_was_training = bool(parent.training)
    facet_was_training = bool(facet.training)
    parent.eval()
    facet.eval()

    parent_logits = {k:[] for k in [*CFG["binary_tasks"], "language", "target"]}
    facet_probs, selectors, views, ids = [], [], [], []

    th = parent.text_encoder.register_forward_hook(_text_hook)
    ih = parent.image_encoder.register_forward_hook(_image_hook)

    for batch in val_loader:
        _CAPTURE.clear()
        with torch.autocast("cuda", dtype=torch.float16):
            outputs = parent(
                batch["input_ids"],
                batch["attention_mask"],
                batch["pixel_values"],
            )
            fused = outputs["fused"]

        zG = parent_target_logodds(fused["target"]).detach()
        zStar, extra = facet(
            _CAPTURE["image_native"].to("cuda:0").detach(),
            _CAPTURE["text_native"].to("cuda:0").detach(),
            zG,
        )

        for k,v in fused.items():
            parent_logits[k].append(v.detach().float().cpu())
        facet_probs.append(torch.sigmoid(zStar).float().cpu())
        selectors.append(extra["selector"].float().cpu())
        views.append(extra["view_weights"].float().cpu())
        ids.extend(list(batch["image"]))

    th.remove(); ih.remove()

    parent_logits = {k:torch.cat(v,0) for k,v in parent_logits.items()}
    parent_probs = logits_to_probs(parent_logits)
    parent_report, _ = metric_report(y_val, parent_probs, raw_thresholds())
    facet_probs = torch.cat(facet_probs,0).numpy()

    result = {
        "ids": ids,
        "parent_report": parent_report,
        "facet_probs": facet_probs,
        "target_report": target_metric_bundle(
            y_val["target"], facet_probs, np.full(7,0.5)
        ),
        "selector_mean": float(torch.cat(selectors,0).mean()),
        "view_weight_mean": torch.cat(views,0).mean(dim=0).tolist(),
    }

    parent.train(parent_was_training)
    facet.train(facet_was_training)
    return result

def gated_selection_key(report, step):
    five = report["aggregate_5task"]["macro_f1"]
    six = report["aggregate_6task"]["macro_f1"]
    six_acc = report["aggregate_6task"]["accuracy"]
    gap_pp = abs(six_acc-six)*100.0
    return (five, six, -gap_pp, six_acc, -int(step))

def save_candidate_pair(seed, ctx, parent, facet, step, native_eval, mode):
    gp = ctx["gated_candidates"] / f"checkpoint-{step:04d}.pt"
    fp = ctx["target_candidates"] / f"checkpoint-{step:04d}.pt"

    torch.save({
        "schema": "M3Offend_BanglaBERT_FacetGate_GatedCandidate_v1_0_1",
        "run_seed": int(seed),
        "global_step": int(step),
        "execution_mode": mode,
        "text_model": CFG["text_model"],
        "vision_model": CFG["vision_model"],
        "text_revision": CFG["text_revision"],
        "vision_revision": CFG["vision_revision"],
        "dataset_sha256": CFG["expected_sha256"],
        "protocol_sha256": EXPECTED_PROTOCOL_SHA256,
        "binding_sha256": FACET_BINDING_SHA256,
        "trainable_state": trainable_state(parent),
        "validation_raw_report": native_eval["parent_report"],
        "selection_key": list(gated_selection_key(native_eval["parent_report"], step)),
        "test_forward_count": 0,
    }, gp)

    torch.save({
        "schema": "M3Offend_BanglaBERT_FacetGate_TargetCandidate_v1_0_1",
        "run_seed": int(seed),
        "global_step": int(step),
        "execution_mode": mode,
        "protocol_sha256": EXPECTED_PROTOCOL_SHA256,
        "binding_sha256": FACET_BINDING_SHA256,
        "state_dict": {k:v.detach().cpu().clone() for k,v in facet.state_dict().items()},
        "native_pair_target_report": native_eval["target_report"],
        "test_forward_count": 0,
    }, fp)
    return gp, fp

def instantiate_reused_parent(source):
    cp = torch.load(source["checkpoint"], map_location="cpu", weights_only=False)
    parent = MemeGuardSixTask()
    if "full_state_dict" in cp:
        parent.load_state_dict(cp["full_state_dict"], strict=True)
    elif "state_dict" in cp:
        parent.load_state_dict(cp["state_dict"], strict=True)
    elif "trainable_state" in cp:
        load_trainable_state(parent, cp["trainable_state"])
    else:
        raise RuntimeError("Reusable parent checkpoint has no recognized model state.")
    return parent, cp

def run_simultaneous_seed(seed, ctx):
    seed_everything(seed)

    parent = MemeGuardSixTask()
    facet = make_facet_model(parent, seed)

    parent_opt, parent_sched, image_params, text_params, fusion_params = make_parent_optimizer(parent)
    facet_opt, facet_sched = make_facet_optimizer(facet)
    parent_scaler = torch.amp.GradScaler("cuda", enabled=True)
    facet_scaler = torch.amp.GradScaler("cuda", enabled=True)

    parent_params = image_params + text_params + fusion_params
    facet_params = list(facet.parameters())

    th = parent.text_encoder.register_forward_hook(_text_hook)
    ih = parent.image_encoder.register_forward_hook(_image_hook)

    parent_opt.zero_grad(set_to_none=True)
    facet_opt.zero_grad(set_to_none=True)

    candidate_records = []
    sample_manifest = []
    global_step = 0

    for epoch in range(FACET_CFG["epochs"]):
        train_loader, order = make_exact_epoch_loader(epoch, seed)
        image_order = train_df.iloc[order]["image"].astype(str).tolist()
        sample_manifest.append({
            "epoch": epoch+1,
            "rows": int(len(order)),
            "unique_rows": int(len(np.unique(order))),
            "order_sha256": hashlib.sha256(
                ("\n".join(image_order)+"\n").encode("utf-8")
            ).hexdigest(),
        })

        parent.train()
        facet.train()
        group_real = 0
        group_micro = 0

        pbar = tqdm(train_loader, desc=f"FacetGate seed {seed} epoch {epoch+1}/2")

        for micro_idx,batch in enumerate(pbar, start=1):
            labels = {k:v.to("cuda:0", non_blocking=True) for k,v in batch["labels"].items()}
            real_n = int(labels["target"].shape[0])
            group_real += real_n
            group_micro += 1

            # One parent image/text forward per microbatch.
            _CAPTURE.clear()
            with torch.autocast("cuda", dtype=torch.float16):
                outputs = parent(
                    batch["input_ids"],
                    batch["attention_mask"],
                    batch["pixel_values"],
                )
                parent_loss_sum = gated_loss_scalar(outputs, labels) * real_n

            if "text_native" not in _CAPTURE or "image_native" not in _CAPTURE:
                raise RuntimeError("Single-forward feature capture failed.")

            zG = parent_target_logodds(outputs["fused"]["target"]).detach()

            with torch.autocast("cuda", dtype=torch.float16):
                zStar, extra = facet(
                    _CAPTURE["image_native"].to("cuda:0").detach(),
                    _CAPTURE["text_native"].to("cuda:0").detach(),
                    zG,
                )
                facet_loss_sum = facet_loss_vector(
                    zStar, extra, zG, labels["target"].float().detach()
                ).sum()

            parent_scaler.scale(parent_loss_sum).backward()
            facet_scaler.scale(facet_loss_sum).backward()

            group_end = (
                group_micro == FACET_CFG["gradient_accumulation_microbatches"]
                or micro_idx == len(train_loader)
            )

            if group_end:
                parent_scaler.unscale_(parent_opt)
                facet_scaler.unscale_(facet_opt)

                divide_gradients(parent_params, group_real)
                divide_gradients(facet_params, group_real)

                torch.nn.utils.clip_grad_norm_(image_params, CFG["gradient_clip"])
                torch.nn.utils.clip_grad_norm_(text_params+fusion_params, CFG["gradient_clip"])
                torch.nn.utils.clip_grad_norm_(facet_params, FACET_CFG["max_grad_norm"])

                parent_scaler.step(parent_opt)
                facet_scaler.step(facet_opt)
                parent_scaler.update()
                facet_scaler.update()
                parent_sched.step()
                facet_sched.step()

                parent_opt.zero_grad(set_to_none=True)
                facet_opt.zero_grad(set_to_none=True)

                global_step += 1

                if global_step in {466,932} and group_real != 14:
                    raise RuntimeError(
                        f"Seed {seed}: final partial group step {global_step} "
                        f"has {group_real} examples, expected 14."
                    )

                if global_step in FACET_CFG["candidate_steps"]:
                    ev = evaluate_native_pair(parent, facet)
                    gp,fp = save_candidate_pair(
                        seed, ctx, parent, facet, global_step, ev,
                        "SIMULTANEOUS_FROM_SCRATCH"
                    )
                    candidate_records.append({
                        "step": global_step,
                        "gated_checkpoint": str(gp),
                        "target_checkpoint": str(fp),
                        "gated_selection_key": list(
                            gated_selection_key(ev["parent_report"], global_step)
                        ),
                        "gated_5task_accuracy": ev["parent_report"]["aggregate_5task"]["accuracy"],
                        "gated_5task_macro_f1": ev["parent_report"]["aggregate_5task"]["macro_f1"],
                        "gated_6task_accuracy": ev["parent_report"]["aggregate_6task"]["accuracy"],
                        "gated_6task_macro_f1": ev["parent_report"]["aggregate_6task"]["macro_f1"],
                        "native_pair_target_macro_f1": ev["target_report"]["macro_f1"],
                        "native_pair_target_map": ev["target_report"]["map"],
                    })

                group_real = 0
                group_micro = 0

    th.remove(); ih.remove()

    if global_step != 932:
        raise RuntimeError(f"Seed {seed}: optimizer steps {global_step} != 932.")
    if [r["step"] for r in candidate_records] != FACET_CFG["candidate_steps"]:
        raise RuntimeError(f"Seed {seed}: candidate inventory mismatch.")

    pd.DataFrame(sample_manifest).to_csv(ctx["facet"]/ "seed_sample_order_manifest.csv", index=False)
    pd.DataFrame(candidate_records).to_csv(ctx["facet"]/ "native_pair_candidate_metrics.csv", index=False)

    selected = max(candidate_records, key=lambda r:tuple(r["gated_selection_key"]))
    cp = torch.load(selected["gated_checkpoint"], map_location="cpu", weights_only=False)
    load_trainable_state(parent, cp["trainable_state"])
    parent.eval()
    for p in parent.parameters():
        p.requires_grad = False

    return {
        "mode": "SIMULTANEOUS_FROM_SCRATCH",
        "parent": parent,
        "selected_gated_record": selected,
        "target_candidate_paths": {int(r["step"]):r["target_checkpoint"] for r in candidate_records},
        "attached_thresholds": None,
    }

def run_reuse_seed(seed, ctx, source):
    seed_everything(seed)
    parent,_ = instantiate_reused_parent(source)
    parent.eval()
    for p in parent.parameters():
        p.requires_grad = False

    attached_thresholds = json.loads(source["thresholds"].read_text(encoding="utf-8"))
    facet = make_facet_model(parent, seed)
    facet_opt,facet_sched = make_facet_optimizer(facet)
    facet_scaler = torch.amp.GradScaler("cuda", enabled=True)
    facet_params = list(facet.parameters())

    th = parent.text_encoder.register_forward_hook(_text_hook)
    ih = parent.image_encoder.register_forward_hook(_image_hook)

    facet_opt.zero_grad(set_to_none=True)
    global_step = 0
    candidate_records = []
    sample_manifest = []

    for epoch in range(FACET_CFG["epochs"]):
        train_loader,order = make_exact_epoch_loader(epoch,seed)
        image_order = train_df.iloc[order]["image"].astype(str).tolist()
        sample_manifest.append({
            "epoch":epoch+1,
            "rows":int(len(order)),
            "unique_rows":int(len(np.unique(order))),
            "order_sha256":hashlib.sha256(
                ("\n".join(image_order)+"\n").encode("utf-8")
            ).hexdigest(),
        })

        facet.train()
        group_real=0
        group_micro=0
        pbar=tqdm(train_loader,desc=f"FacetGate reuse seed {seed} epoch {epoch+1}/2")

        for micro_idx,batch in enumerate(pbar,start=1):
            labels={k:v.to("cuda:0",non_blocking=True) for k,v in batch["labels"].items()}
            real_n=int(labels["target"].shape[0])
            group_real += real_n
            group_micro += 1

            _CAPTURE.clear()
            with torch.no_grad(), torch.autocast("cuda",dtype=torch.float16):
                outputs=parent(
                    batch["input_ids"],
                    batch["attention_mask"],
                    batch["pixel_values"],
                )

            zG=parent_target_logodds(outputs["fused"]["target"]).detach()
            with torch.autocast("cuda",dtype=torch.float16):
                zStar,extra=facet(
                    _CAPTURE["image_native"].to("cuda:0").detach(),
                    _CAPTURE["text_native"].to("cuda:0").detach(),
                    zG,
                )
                loss_sum=facet_loss_vector(
                    zStar,extra,zG,labels["target"].float().detach()
                ).sum()

            facet_scaler.scale(loss_sum).backward()

            group_end=(
                group_micro == FACET_CFG["gradient_accumulation_microbatches"]
                or micro_idx == len(train_loader)
            )
            if group_end:
                facet_scaler.unscale_(facet_opt)
                divide_gradients(facet_params,group_real)
                torch.nn.utils.clip_grad_norm_(facet_params,FACET_CFG["max_grad_norm"])
                facet_scaler.step(facet_opt)
                facet_scaler.update()
                facet_sched.step()
                facet_opt.zero_grad(set_to_none=True)

                global_step += 1
                if global_step in {466,932} and group_real != 14:
                    raise RuntimeError(
                        f"Seed {seed}: final partial group {group_real} != 14."
                    )

                if global_step in FACET_CFG["candidate_steps"]:
                    ev=evaluate_native_pair(parent,facet)
                    fp=ctx["target_candidates"]/f"checkpoint-{global_step:04d}.pt"
                    torch.save({
                        "schema":"M3Offend_BanglaBERT_FacetGate_TargetCandidate_v1_0_1",
                        "run_seed":int(seed),
                        "global_step":int(global_step),
                        "execution_mode":"REUSE_EXISTING_GATED",
                        "protocol_sha256":EXPECTED_PROTOCOL_SHA256,
                        "binding_sha256":FACET_BINDING_SHA256,
                        "state_dict":{k:v.detach().cpu().clone() for k,v in facet.state_dict().items()},
                        "native_pair_target_report":ev["target_report"],
                        "test_forward_count":0,
                    },fp)
                    candidate_records.append({
                        "step":global_step,
                        "target_checkpoint":str(fp),
                        "native_pair_target_macro_f1":ev["target_report"]["macro_f1"],
                        "native_pair_target_map":ev["target_report"]["map"],
                    })

                group_real=0
                group_micro=0

    th.remove(); ih.remove()

    if global_step != 932:
        raise RuntimeError(f"Seed {seed}: FacetGate steps {global_step} != 932.")

    pd.DataFrame(sample_manifest).to_csv(ctx["facet"]/ "seed_sample_order_manifest.csv",index=False)
    pd.DataFrame(candidate_records).to_csv(ctx["facet"]/ "native_pair_candidate_metrics.csv",index=False)

    return {
        "mode":"REUSE_EXISTING_GATED",
        "parent":parent,
        "selected_gated_record":{
            "step":source["manifest"].get("selected_step"),
            "source_manifest":str(source["manifest_path"]),
        },
        "target_candidate_paths":{int(r["step"]):r["target_checkpoint"] for r in candidate_records},
        "attached_thresholds":attached_thresholds,
    }

In [ ]:
# Per-seed checkpoint finalization, validation calibration, bootstrap, and freeze.

def calibrate_facetgate_target(y, probs):
    grid=np.asarray([round(0.05+0.025*i,3) for i in range(37)],dtype=float)
    thresholds=[]
    audit=[]
    for j,lab in enumerate(CFG["targets"]):
        yy=np.asarray(y[:,j],dtype=int)
        pp=np.asarray(probs[:,j],dtype=float)
        support=int(yy.sum())
        f0=float(f1_score(yy,(pp>=0.5).astype(int),zero_division=0))
        scored=[
            (float(f1_score(yy,(pp>=th).astype(int),zero_division=0)),float(th))
            for th in grid
        ]
        best=max(x[0] for x in scored)
        tied=[th for f,th in scored if abs(f-best)<=1e-12]
        chosen=sorted(tied,key=lambda x:(abs(x-0.5),x))[0]
        if support < 5 or best-f0 < 0.005:
            chosen=0.5
        thresholds.append(float(chosen))
        audit.append({
            "label":lab,
            "support":support,
            "f1_0p5":f0,
            "best_grid_f1":best,
            "selected_threshold":float(chosen),
        })
    return np.asarray(thresholds),audit

def _fixed_cache_for_parent(parent):
    capture={}
    def thook(module,inputs,output):
        capture["text"]=output.last_hidden_state[:,0].detach()
    def ihook(module,inputs,output):
        capture["image"]=output.pooler_output.detach()

    th=parent.text_encoder.register_forward_hook(thook)
    ih=parent.image_encoder.register_forward_hook(ihook)

    ids=[]
    text_native=[]
    image_native=[]
    zG=[]
    logits={k:[] for k in [*CFG["binary_tasks"],"language","target"]}

    parent.eval()
    with torch.no_grad():
        for batch in tqdm(val_loader,desc="selected Gated validation cache",leave=False):
            capture.clear()
            with torch.autocast("cuda",dtype=torch.float16):
                outputs=parent(
                    batch["input_ids"],
                    batch["attention_mask"],
                    batch["pixel_values"],
                )
                fused=outputs["fused"]

            text_native.append(capture["text"].float().cpu())
            image_native.append(capture["image"].float().cpu())
            zG.append(parent_target_logodds(fused["target"]).float().cpu())
            for k,v in fused.items():
                logits[k].append(v.detach().float().cpu())
            ids.extend(list(batch["image"]))

    th.remove(); ih.remove()

    if ids != val_df["image"].astype(str).tolist():
        raise RuntimeError("Validation cache row-order drift.")

    return {
        "ids":ids,
        "text_native":torch.cat(text_native,0),
        "image_native":torch.cat(image_native,0),
        "zG":torch.cat(zG,0),
        "logits":{k:torch.cat(v,0) for k,v in logits.items()},
    }

def _replay_target_candidate(seed,parent,fixed,path):
    cp=torch.load(path,map_location="cpu",weights_only=False)
    facet=make_facet_model(parent,seed)
    facet.load_state_dict(cp["state_dict"],strict=True)
    facet.eval()

    probs=[]
    selectors=[]
    views=[]
    with torch.no_grad():
        for start in range(0,932,128):
            sl=slice(start,min(start+128,932))
            zStar,extra=facet(
                fixed["image_native"][sl].to("cuda:0"),
                fixed["text_native"][sl].to("cuda:0"),
                fixed["zG"][sl].to("cuda:0"),
            )
            probs.append(torch.sigmoid(zStar).float().cpu())
            selectors.append(extra["selector"].float().cpu())
            views.append(extra["view_weights"].float().cpu())

    out=(
        torch.cat(probs,0).numpy(),
        torch.cat(selectors,0).numpy(),
        torch.cat(views,0).numpy(),
    )
    del facet
    torch.cuda.empty_cache()
    return out

def target_macro_from_pred(y,pred):
    return float(f1_score(y,pred,average="macro",zero_division=0))

def finalize_seed_development(seed,ctx,run):
    parent=run["parent"]
    parent.eval()
    for p in parent.parameters():
        p.requires_grad=False

    fixed=_fixed_cache_for_parent(parent)

    fixed_cache_manifest = {
        "seed": int(seed),
        "rows": int(len(fixed["ids"])),
        "ordered_image_id_sha256": hashlib.sha256(
            ("\n".join(map(str, fixed["ids"])) + "\n").encode("utf-8")
        ).hexdigest(),
        "text_native_shape": list(fixed["text_native"].shape),
        "image_native_shape": list(fixed["image_native"].shape),
        "zG_shape": list(fixed["zG"].shape),
        "test_forward_count": 0,
    }
    (ctx["facet"] / "FIXED_SELECTED_GATED_VALIDATION_CACHE_MANIFEST.json").write_text(
        json.dumps(fixed_cache_manifest, indent=2),
        encoding="utf-8",
    )

    gated_probs=logits_to_probs(fixed["logits"])
    gated_raw,_=metric_report(y_val,gated_probs,raw_thresholds())

    if run["attached_thresholds"] is None:
        gated_thresholds,gated_audit=calibrate_thresholds_after_selection(
            y_val,gated_probs
        )
    else:
        gated_thresholds=run["attached_thresholds"]
        gated_audit={"source":"attached frozen parent validation thresholds"}

    gated_cal,_=metric_report(y_val,gated_probs,gated_thresholds)

    # Freeze the selected Gated checkpoint before Target replay.
    selected_gated_path=ctx["facet"]/ "selected_gated_checkpoint.pt"
    torch.save({
        "schema":"M3Offend_BanglaBERT_FacetGate_SelectedGated_v1_0_1",
        "run_seed":int(seed),
        "selected_step":run["selected_gated_record"].get("step"),
        "text_model":CFG["text_model"],
        "vision_model":CFG["vision_model"],
        "text_revision":CFG["text_revision"],
        "vision_revision":CFG["vision_revision"],
        "dataset_sha256":CFG["expected_sha256"],
        "protocol_sha256":EXPECTED_PROTOCOL_SHA256,
        "binding_sha256":FACET_BINDING_SHA256,
        "full_state_dict":{k:v.detach().cpu() for k,v in parent.state_dict().items()},
        "validation_raw_report":gated_raw,
        "validation_calibrated_report":gated_cal,
        "test_forward_count":0,
    },selected_gated_path)

    (ctx["facet"]/ "selected_gated_thresholds.json").write_text(
        json.dumps(gated_thresholds,indent=2),encoding="utf-8"
    )
    (ctx["facet"]/ "selected_gated_threshold_audit.json").write_text(
        json.dumps(gated_audit,indent=2),encoding="utf-8"
    )

    # Replay all five saved FacetGate candidates against the selected Gated checkpoint.
    replay_rows=[]
    replay_outputs={}
    for step in FACET_CFG["candidate_steps"]:
        path=Path(run["target_candidate_paths"][step])
        cp=torch.load(path,map_location="cpu",weights_only=False)
        if int(cp["test_forward_count"]) != 0:
            raise RuntimeError(f"Seed {seed}: Target candidate already touched TEST.")
        probs,selectors,views=_replay_target_candidate(seed,parent,fixed,path)
        metrics=target_metric_bundle(y_val["target"],probs,np.full(7,0.5))
        row={
            "step":int(step),
            "target_macro_f1_raw_0p5":metrics["macro_f1"],
            "target_map":metrics["map"],
            "target_exact_set_accuracy_raw_0p5":metrics["exact_set_accuracy"],
            "selection_key":[
                metrics["macro_f1"],
                metrics["map"],
                metrics["exact_set_accuracy"],
                -int(step),
            ],
        }
        replay_rows.append(row)
        replay_outputs[step]={
            "probs":probs,
            "metrics":metrics,
            "selector_mean":float(selectors.mean()),
            "view_weight_mean":views.mean(axis=0).tolist(),
        }

    pd.DataFrame(replay_rows).to_csv(
        ctx["facet"]/ "FACETGATE_COMPATIBILITY_REPLAY_TABLE.csv",index=False
    )

    selected_target=max(replay_rows,key=lambda r:tuple(r["selection_key"]))
    selected_target_step=int(selected_target["step"])
    selected_target_probs=replay_outputs[selected_target_step]["probs"]

    target_thresholds,target_audit=calibrate_facetgate_target(
        y_val["target"],selected_target_probs
    )

    parent_target_thresholds=np.asarray(
        [gated_thresholds["target"][lab] for lab in CFG["targets"]],dtype=float
    )
    parent_target=target_metric_bundle(
        y_val["target"],gated_probs["target"],parent_target_thresholds
    )
    facet_target=target_metric_bundle(
        y_val["target"],selected_target_probs,target_thresholds
    )

    composite_probs={
        **{t:gated_probs[t] for t in CFG["binary_tasks"]},
        "language":gated_probs["language"],
        "target":selected_target_probs,
    }
    composite_thresholds={
        **{t:gated_thresholds[t] for t in CFG["binary_tasks"]},
        "target":{
            lab:float(target_thresholds[j])
            for j,lab in enumerate(CFG["targets"])
        },
    }
    composite_report,_=metric_report(y_val,composite_probs,composite_thresholds)

    # Verify exact Core-5 identity.
    max_diff=0.0
    for task in [*CFG["binary_tasks"],"language"]:
        max_diff=max(
            max_diff,
            float(np.max(np.abs(
                np.asarray(composite_probs[task])-np.asarray(gated_probs[task])
            )))
        )
    if max_diff > 1e-10:
        raise RuntimeError(f"Seed {seed}: Core-5 identity violation {max_diff}.")
    if composite_report["aggregate_5task"] != gated_cal["aggregate_5task"]:
        raise RuntimeError(f"Seed {seed}: 5-task identity violation.")

    # Save the selected Target checkpoint.
    selected_cp=torch.load(
        run["target_candidate_paths"][selected_target_step],
        map_location="cpu",
        weights_only=False,
    )
    selected_target_path=ctx["facet"]/ "selected_facetgate_target_checkpoint.pt"
    torch.save({
        "schema":"M3Offend_BanglaBERT_FacetGate_SelectedTarget_v1_0_1",
        "run_seed":int(seed),
        "selected_target_step":selected_target_step,
        "selected_gated_step":run["selected_gated_record"].get("step"),
        "protocol_sha256":EXPECTED_PROTOCOL_SHA256,
        "binding_sha256":FACET_BINDING_SHA256,
        "state_dict":selected_cp["state_dict"],
        "target_thresholds":{
            lab:float(target_thresholds[j])
            for j,lab in enumerate(CFG["targets"])
        },
        "test_forward_count":0,
    },selected_target_path)

    (ctx["facet"]/ "facetgate_target_thresholds.json").write_text(
        json.dumps(
            {lab:float(target_thresholds[j]) for j,lab in enumerate(CFG["targets"])},
            indent=2
        ),encoding="utf-8"
    )
    (ctx["facet"]/ "facetgate_target_threshold_audit.json").write_text(
        json.dumps(target_audit,indent=2),encoding="utf-8"
    )

    # Compute the final validation comparison.
    side=[]
    for task in CFG["binary_tasks"]+["language"]:
        side.append({
            "group":"Core5",
            "task_or_label":task,
            "Gated_accuracy_pct":100*gated_cal[task]["accuracy"],
            "Gated_F1_pct":100*gated_cal[task]["macro_f1"],
            "FacetGate_accuracy_pct":100*composite_report[task]["accuracy"],
            "FacetGate_F1_pct":100*composite_report[task]["macro_f1"],
            "delta_F1_pp":0.0,
            "status":"UNCHANGED",
        })
    for lab in CFG["targets"]:
        old=parent_target["per_label_f1"][lab]
        new=facet_target["per_label_f1"][lab]
        d=100*(new-old)
        side.append({
            "group":"TargetLabel",
            "task_or_label":lab,
            "Gated_accuracy_pct":np.nan,
            "Gated_F1_pct":100*old,
            "FacetGate_accuracy_pct":np.nan,
            "FacetGate_F1_pct":100*new,
            "delta_F1_pp":d,
            "status":"IMPROVED" if d>1e-12 else ("WORSE" if d<-1e-12 else "UNCHANGED"),
        })
    pd.DataFrame(side).to_csv(
        ctx["facet"]/ "FACETGATE_FULL_SIDE_BY_SIDE_VALIDATION.csv",index=False
    )

    # Paired 10,000-repeat item and pHash-group bootstrap.
    old_pred=(gated_probs["target"]>=parent_target_thresholds[None,:]).astype(int)
    new_pred=(selected_target_probs>=target_thresholds[None,:]).astype(int)

    B=10000
    rng=np.random.default_rng(int(seed)+20260911)
    item_deltas=np.empty(B,dtype=np.float32)
    for b in range(B):
        idx=rng.integers(0,932,size=932)
        item_deltas[b]=(
            target_macro_from_pred(y_val["target"][idx],new_pred[idx])
            - target_macro_from_pred(y_val["target"][idx],old_pred[idx])
        )

    unique_groups=np.unique(VAL_BOOTSTRAP_GROUPS)
    positions={g:np.where(VAL_BOOTSTRAP_GROUPS==g)[0] for g in unique_groups}
    rngg=np.random.default_rng(int(seed)+20260912)
    group_deltas=np.empty(B,dtype=np.float32)
    for b in range(B):
        sampled=rngg.choice(unique_groups,size=len(unique_groups),replace=True)
        idx=np.concatenate([positions[g] for g in sampled])
        group_deltas[b]=(
            target_macro_from_pred(y_val["target"][idx],new_pred[idx])
            - target_macro_from_pred(y_val["target"][idx],old_pred[idx])
        )

    bootstrap={
        "item_ci95":[float(np.quantile(item_deltas,0.025)),float(np.quantile(item_deltas,0.975))],
        "group_ci95":[float(np.quantile(group_deltas,0.025)),float(np.quantile(group_deltas,0.975))],
        "item_probability_positive":float(np.mean(item_deltas>0)),
        "group_probability_positive":float(np.mean(group_deltas>0)),
        "repeats":B,
    }
    (ctx["facet"]/ "facetgate_paired_bootstrap.json").write_text(
        json.dumps(bootstrap,indent=2),encoding="utf-8"
    )

    summary={
        "seed":int(seed),
        "execution_mode":run["mode"],
        "selected_gated_step":run["selected_gated_record"].get("step"),
        "selected_target_step":selected_target_step,
        "validation_gated_5task":gated_cal["aggregate_5task"],
        "validation_facetgate_5task":composite_report["aggregate_5task"],
        "validation_gated_target":parent_target,
        "validation_facetgate_target":facet_target,
        "validation_gated_6task":gated_cal["aggregate_6task"],
        "validation_facetgate_6task":composite_report["aggregate_6task"],
        "target_delta_macro_f1_pp":100*(facet_target["macro_f1"]-parent_target["macro_f1"]),
        "target_delta_map_pp":100*(facet_target["map"]-parent_target["map"]),
        "core5_max_abs_diff":max_diff,
        "selected_gated_checkpoint":str(selected_gated_path),
        "selected_target_checkpoint":str(selected_target_path),
        "gated_thresholds_file":str(ctx["facet"]/ "selected_gated_thresholds.json"),
        "target_thresholds_file":str(ctx["facet"]/ "facetgate_target_thresholds.json"),
        "bootstrap":bootstrap,
        "development_test_forward_count":0,
    }

    (ctx["facet"]/ "FACETGATE_FINAL_DEVELOPMENT_RESULT.json").write_text(
        json.dumps(summary,indent=2),encoding="utf-8"
    )
    (ctx["facet"]/ "DEVELOPMENT_COMPLETE.json").write_text(
        json.dumps({
            "status":"DEVELOPMENT_COMPLETE_TEST_LOCKED",
            "seed":int(seed),
            "execution_mode":run["mode"],
            "selected_gated_checkpoint":str(selected_gated_path),
            "selected_target_checkpoint":str(selected_target_path),
            "gated_thresholds_file":summary["gated_thresholds_file"],
            "target_thresholds_file":summary["target_thresholds_file"],
            "test_forward_count":0,
        },indent=2),encoding="utf-8"
    )

    # Release GPU memory before the next seed.
    # Remove retained model references before cleanup.
    run["parent"] = None
    del fixed
    del parent
    gc.collect()
    torch.cuda.empty_cache()

    return summary

In [ ]:
# Run all five development seeds.
ALL_DEVELOPMENT = []

for seed in PUBLICATION_SEEDS:
    print("\n" + "#"*110)
    print(f"FACETGATE DEVELOPMENT SEED {seed}")
    print("#"*110)

    RUN_SEED = int(seed)
    CFG["run_seed"] = int(seed)

    ctx = seed_paths(seed)
    mode,parent_source,rejected = discover_parent_for_seed(seed)

    startup={
        "seed":int(seed),
        "mode":mode,
        "rejected_parent_candidates":rejected,
        "development_TEST_forward_count":0,
    }
    (ctx["facet"]/ "startup_provenance.json").write_text(
        json.dumps(startup,indent=2),encoding="utf-8"
    )

    if mode == "REUSE_EXISTING_GATED":
        run = run_reuse_seed(seed,ctx,parent_source)
    else:
        run = run_simultaneous_seed(seed,ctx)

    dev = finalize_seed_development(seed,ctx,run)
    ALL_DEVELOPMENT.append(dev)

    del run
    gc.collect()
    torch.cuda.empty_cache()

    print(
        f"[SEED {seed} DEVELOPMENT COMPLETE] "
        f"5-task MF1={100*dev['validation_facetgate_5task']['macro_f1']:.3f}% | "
        f"Target MF1={100*dev['validation_facetgate_target']['macro_f1']:.3f}% | "
        f"Target Δ={dev['target_delta_macro_f1_pp']:+.3f} pp | "
        f"6-task MF1={100*dev['validation_facetgate_6task']['macro_f1']:.3f}%"
    )

if [d["seed"] for d in ALL_DEVELOPMENT] != PUBLICATION_SEEDS:
    raise RuntimeError("Not all five development seeds completed.")

if any(int(d["development_test_forward_count"]) != 0 for d in ALL_DEVELOPMENT):
    raise RuntimeError("TEST was accessed during development.")

development_modes = sorted(set(d["execution_mode"] for d in ALL_DEVELOPMENT))
if len(development_modes) != 1:
    raise RuntimeError(
        "Mixed FacetGate execution modes across publication seeds are not allowed "
        f"for one five-seed aggregate: {development_modes}. "
        "Attach either no reusable parents (all simultaneous) or valid reusable "
        "parents for all five seeds."
    )
print("ALL-FIVE EXECUTION FAMILY:", development_modes[0])

pd.DataFrame([
    {
        "seed":d["seed"],
        "mode":d["execution_mode"],
        "selected_gated_step":d["selected_gated_step"],
        "selected_target_step":d["selected_target_step"],
        "val_5task_accuracy":d["validation_facetgate_5task"]["accuracy"],
        "val_5task_macro_f1":d["validation_facetgate_5task"]["macro_f1"],
        "val_target_macro_f1":d["validation_facetgate_target"]["macro_f1"],
        "val_target_map":d["validation_facetgate_target"]["map"],
        "val_6task_accuracy":d["validation_facetgate_6task"]["accuracy"],
        "val_6task_macro_f1":d["validation_facetgate_6task"]["macro_f1"],
        "target_delta_pp":d["target_delta_macro_f1_pp"],
    }
    for d in ALL_DEVELOPMENT
]).to_csv(ALL5_ROOT/"ALL5_DEVELOPMENT_SUMMARY.csv",index=False)

print("\nALL FIVE DEVELOPMENT RUNS COMPLETE.")
print("LOCKED TEST HAS NOT BEEN FORWARDED YET.")

In [ ]:
# Freeze all selected states and thresholds before locked-test evaluation.
def sha256_file(path,chunk=1<<20):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

freeze_rows=[]
for d in ALL_DEVELOPMENT:
    required=[
        d["selected_gated_checkpoint"],
        d["selected_target_checkpoint"],
        d["gated_thresholds_file"],
        d["target_thresholds_file"],
    ]
    for p in required:
        if not Path(p).is_file():
            raise FileNotFoundError(p)

    freeze_rows.append({
        "seed":d["seed"],
        "selected_gated_checkpoint":d["selected_gated_checkpoint"],
        "selected_gated_sha256":sha256_file(d["selected_gated_checkpoint"]),
        "selected_target_checkpoint":d["selected_target_checkpoint"],
        "selected_target_sha256":sha256_file(d["selected_target_checkpoint"]),
        "gated_thresholds_sha256":sha256_file(d["gated_thresholds_file"]),
        "target_thresholds_sha256":sha256_file(d["target_thresholds_file"]),
        "development_test_forward_count":0,
    })

FREEZE_DF=pd.DataFrame(freeze_rows)
FREEZE_DF.to_csv(ALL5_ROOT/"DEVELOPMENT_FREEZE_MANIFEST_BEFORE_TEST.csv",index=False)

FREEZE_BY_SEED = {
    int(row["seed"]): row
    for row in freeze_rows
}

freeze_payload=FREEZE_DF.to_json(orient="records",double_precision=15)
DEVELOPMENT_FREEZE_SHA256=hashlib.sha256(freeze_payload.encode("utf-8")).hexdigest()

(ALL5_ROOT/"DEVELOPMENT_FREEZE_MANIFEST.json").write_text(
    json.dumps({
        "status":"ALL_FIVE_DEVELOPMENT_STATES_FROZEN_BEFORE_TEST",
        "seeds":PUBLICATION_SEEDS,
        "freeze_sha256":DEVELOPMENT_FREEZE_SHA256,
        "development_test_forward_count":0,
    },indent=2),encoding="utf-8"
)

DEVELOPMENT_PHASE_FROZEN=True
print("DEVELOPMENT FREEZE BARRIER PASS")
print("Freeze SHA-256:",DEVELOPMENT_FREEZE_SHA256)
print("Now, and only now, locked TEST is authorized.")

In [ ]:
# Final locked-test evaluation: one complete pass per frozen seed.
if not DEVELOPMENT_PHASE_FROZEN:
    raise RuntimeError("Locked TEST cannot run before all-five development freeze.")

if len(test_df) != 932:
    raise RuntimeError(f"Locked TEST row count {len(test_df)} != 932.")

y_test = frame_truth(test_df)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG["batch_size"] * 4,
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True,
)

def load_frozen_seed_models(dev):
    seed = int(dev["seed"])
    if seed not in FREEZE_BY_SEED:
        raise RuntimeError(f"Seed {seed}: missing from development freeze manifest.")

    frozen = FREEZE_BY_SEED[seed]
    current_hashes = {
        "selected_gated_sha256": sha256_file(dev["selected_gated_checkpoint"]),
        "selected_target_sha256": sha256_file(dev["selected_target_checkpoint"]),
        "gated_thresholds_sha256": sha256_file(dev["gated_thresholds_file"]),
        "target_thresholds_sha256": sha256_file(dev["target_thresholds_file"]),
    }
    for key, observed in current_hashes.items():
        expected = str(frozen[key])
        if observed != expected:
            raise RuntimeError(
                f"Seed {seed}: frozen artifact changed before TEST: {key}; "
                f"expected={expected}, observed={observed}"
            )

    gated_cp=torch.load(
        dev["selected_gated_checkpoint"],map_location="cpu",weights_only=False
    )
    parent=MemeGuardSixTask()
    parent.load_state_dict(gated_cp["full_state_dict"],strict=True)
    del gated_cp
    parent.eval()
    for p in parent.parameters():
        p.requires_grad=False

    target_cp=torch.load(
        dev["selected_target_checkpoint"],map_location="cpu",weights_only=False
    )
    facet=make_facet_model(parent,dev["seed"])
    facet.load_state_dict(target_cp["state_dict"],strict=True)
    del target_cp
    facet.eval()

    gated_thresholds=json.loads(
        Path(dev["gated_thresholds_file"]).read_text(encoding="utf-8")
    )
    target_thresholds=json.loads(
        Path(dev["target_thresholds_file"]).read_text(encoding="utf-8")
    )

    return parent,facet,gated_thresholds,target_thresholds

@torch.no_grad()
def locked_test_one_seed(dev):
    global LOCKED_TEST_SEED_PASSES

    seed=int(dev["seed"])
    parent,facet,gated_thresholds,target_thresholds=load_frozen_seed_models(dev)

    capture={}
    def thook(module,inputs,output):
        capture["text"]=output.last_hidden_state[:,0].detach()
    def ihook(module,inputs,output):
        capture["image"]=output.pooler_output.detach()

    th=parent.text_encoder.register_forward_hook(thook)
    ih=parent.image_encoder.register_forward_hook(ihook)

    ids=[]
    core_logits={k:[] for k in [*CFG["binary_tasks"],"language","target"]}
    facet_probs=[]
    selectors=[]
    views=[]

    for batch in tqdm(test_loader,desc=f"LOCKED TEST seed {seed}"):
        capture.clear()

        # FacetGate consumes detached features from the frozen parent forward.
        with torch.autocast("cuda",dtype=torch.float16):
            outputs=parent(
                batch["input_ids"],
                batch["attention_mask"],
                batch["pixel_values"],
            )
            fused=outputs["fused"]

        zG=parent_target_logodds(fused["target"]).detach()

        with torch.autocast("cuda",dtype=torch.float16):
            zStar,extra=facet(
                capture["image"].to("cuda:0").detach(),
                capture["text"].to("cuda:0").detach(),
                zG,
            )

        for k,v in fused.items():
            core_logits[k].append(v.detach().float().cpu())
        facet_probs.append(torch.sigmoid(zStar).float().cpu())
        selectors.append(extra["selector"].float().cpu())
        views.append(extra["view_weights"].float().cpu())
        ids.extend(list(batch["image"]))

    th.remove(); ih.remove()

    if ids != test_df["image"].astype(str).tolist():
        raise RuntimeError(f"Seed {seed}: locked TEST row-order drift.")

    core_logits={k:torch.cat(v,0) for k,v in core_logits.items()}
    probs=logits_to_probs(core_logits)
    probs["target"]=torch.cat(facet_probs,0).numpy()

    thresholds={
        **{t:gated_thresholds[t] for t in CFG["binary_tasks"]},
        "target":target_thresholds,
    }

    report,target_pred=metric_report(y_test,probs,thresholds)
    target_extended=target_metric_bundle(
        y_test["target"],
        probs["target"],
        np.asarray([target_thresholds[lab] for lab in CFG["targets"]],dtype=float),
    )

    seed_test_dir=ALL5_ROOT/f"seed_{seed}"/"locked_test"
    seed_test_dir.mkdir(parents=True,exist_ok=True)

    result={
        "seed":seed,
        "development_freeze_sha256":DEVELOPMENT_FREEZE_SHA256,
        "selected_gated_sha256":sha256_file(dev["selected_gated_checkpoint"]),
        "selected_target_sha256":sha256_file(dev["selected_target_checkpoint"]),
        "metrics":report,
        "target_extended":target_extended,
        "selector_mean":float(torch.cat(selectors,0).mean()),
        "view_weight_mean":torch.cat(views,0).mean(dim=0).tolist(),
        "test_rows":932,
        "test_pass_number":LOCKED_TEST_SEED_PASSES+1,
    }
    (seed_test_dir/f"seed_{seed}_LOCKED_TEST_metrics.json").write_text(
        json.dumps(result,indent=2),encoding="utf-8"
    )

    # Prediction ledger.
    ledger=pd.DataFrame({
        "official_row_index":test_df["official_row_index"].astype(int),
        "image":test_df["image"].astype(str),
        "gold_language":test_df["language"].astype(str),
    })
    for task in CFG["binary_tasks"]:
        ledger[f"gold_{task}"]=y_test[task]
        ledger[f"prob_{task}"]=probs[task]
        ledger[f"pred_{task}"]=(probs[task]>=float(gated_thresholds[task])).astype(int)

    lang_pred=probs["language"].argmax(axis=1)
    ledger["pred_language"]=[CFG["languages"][i] for i in lang_pred]
    for j,lang in enumerate(CFG["languages"]):
        ledger[f"prob_language_{lang}"]=probs["language"][:,j]

    for j,lab in enumerate(CFG["targets"]):
        ledger[f"gold_target__{lab}"]=y_test["target"][:,j]
        ledger[f"prob_target__{lab}"]=probs["target"][:,j]
        ledger[f"pred_target__{lab}"]=target_pred[:,j]

    ledger.to_csv(
        seed_test_dir/f"seed_{seed}_LOCKED_TEST_prediction_ledger.csv",index=False
    )

    # Per-class metrics.
    rows=[]
    for task in CFG["binary_tasks"]:
        pred=(probs[task]>=float(gated_thresholds[task])).astype(int)
        pr,rc,f1v,sup=precision_recall_fscore_support(
            y_test[task],pred,labels=[0,1],zero_division=0
        )
        for cls in [0,1]:
            rows.append({
                "task":task,
                "class":"Yes" if cls==1 else "No",
                "precision":float(pr[cls]),
                "recall":float(rc[cls]),
                "f1":float(f1v[cls]),
                "support":int(sup[cls]),
            })

    lpred=probs["language"].argmax(axis=1)
    pr,rc,f1v,sup=precision_recall_fscore_support(
        y_test["language"],lpred,labels=[0,1,2],zero_division=0
    )
    for j,lab in enumerate(CFG["languages"]):
        rows.append({
            "task":"language","class":lab,
            "precision":float(pr[j]),"recall":float(rc[j]),
            "f1":float(f1v[j]),"support":int(sup[j]),
        })

    for j,lab in enumerate(CFG["targets"]):
        pr,rc,f1v,sup=precision_recall_fscore_support(
            y_test["target"][:,j],target_pred[:,j],labels=[0,1],zero_division=0
        )
        rows.append({
            "task":"target","class":lab,
            "precision":float(pr[1]),"recall":float(rc[1]),
            "f1":float(f1v[1]),"support":int(sup[1]),
            "average_precision":float(target_extended["per_label_ap"][lab]),
        })

    pd.DataFrame(rows).to_csv(
        seed_test_dir/f"seed_{seed}_LOCKED_TEST_per_class_metrics.csv",index=False
    )

    LOCKED_TEST_SEED_PASSES += 1

    print(
        f"[LOCKED TEST seed {seed}] "
        f"5-task Acc={100*report['aggregate_5task']['accuracy']:.3f}% "
        f"MF1={100*report['aggregate_5task']['macro_f1']:.3f}% | "
        f"Target MF1={100*report['target']['macro_f1']:.3f}% "
        f"mAP={100*target_extended['map']:.3f}% | "
        f"6-task Acc={100*report['aggregate_6task']['accuracy']:.3f}% "
        f"MF1={100*report['aggregate_6task']['macro_f1']:.3f}%"
    )

    del parent,facet,core_logits,probs
    gc.collect()
    torch.cuda.empty_cache()

    return result

ALL_TEST_RESULTS=[]
_TESTED_SEEDS = set()

for dev in ALL_DEVELOPMENT:
    seed = int(dev["seed"])
    if seed in _TESTED_SEEDS:
        raise RuntimeError(f"Locked TEST duplicate seed pass blocked: {seed}")
    ALL_TEST_RESULTS.append(locked_test_one_seed(dev))
    _TESTED_SEEDS.add(seed)

if _TESTED_SEEDS != set(PUBLICATION_SEEDS):
    raise RuntimeError(
        f"Locked TEST seed identity mismatch: observed={sorted(_TESTED_SEEDS)}"
    )

if LOCKED_TEST_SEED_PASSES != 5:
    raise RuntimeError(
        f"Expected one locked TEST pass for each of 5 seeds; got {LOCKED_TEST_SEED_PASSES}."
    )

print("ALL FIVE LOCKED TEST SEED PASSES COMPLETE.")

In [ ]:
# Five-seed locked-test aggregation: mean, sample SD, and 95% t-CI.
T_CRIT_DF4 = 2.7764451051977987

per_seed_rows=[]
for r in ALL_TEST_RESULTS:
    m=r["metrics"]
    per_seed_rows.append({
        "seed":r["seed"],
        "5task_accuracy":m["aggregate_5task"]["accuracy"],
        "5task_macro_f1":m["aggregate_5task"]["macro_f1"],
        "target_exact_accuracy":m["target"]["exact_set_accuracy"],
        "target_macro_f1":m["target"]["macro_f1"],
        "target_mAP":r["target_extended"]["map"],
        "6task_accuracy":m["aggregate_6task"]["accuracy"],
        "6task_macro_f1":m["aggregate_6task"]["macro_f1"],
    })

PER_SEED_TEST=pd.DataFrame(per_seed_rows)
PER_SEED_TEST.to_csv(ALL5_ROOT/"FACETGATE_ALL5_LOCKED_TEST_PER_SEED_RAW.csv",index=False)

percent=PER_SEED_TEST.copy()
for c in percent.columns:
    if c!="seed":
        percent[c]=100*percent[c]
percent.to_csv(
    ALL5_ROOT/"FACETGATE_ALL5_LOCKED_TEST_PER_SEED_PERCENT.csv",index=False
)

summary_rows=[]
for metric in [c for c in PER_SEED_TEST.columns if c!="seed"]:
    vals=PER_SEED_TEST[metric].to_numpy(dtype=float)
    mean=float(vals.mean())
    sd=float(vals.std(ddof=1))
    half=T_CRIT_DF4*sd/math.sqrt(5)
    summary_rows.append({
        "metric":metric,
        "mean":mean,
        "sample_sd":sd,
        "ci95_low":mean-half,
        "ci95_high":mean+half,
        "mean_pct":100*mean,
        "sample_sd_pct":100*sd,
        "ci95_low_pct":100*(mean-half),
        "ci95_high_pct":100*(mean+half),
    })

SUMMARY=pd.DataFrame(summary_rows)
SUMMARY.to_csv(
    ALL5_ROOT/"FACETGATE_ALL5_LOCKED_TEST_MEAN_SD_T95CI.csv",index=False
)

# Five-seed summary for each Target label.
target_rows=[]
for r in ALL_TEST_RESULTS:
    seed=r["seed"]
    for lab,f1v in r["metrics"]["target"]["per_label_f1"].items():
        target_rows.append({
            "seed":seed,
            "target_label":lab,
            "f1":float(f1v),
            "average_precision":float(r["target_extended"]["per_label_ap"][lab]),
        })

TARGET_PER_SEED=pd.DataFrame(target_rows)
TARGET_PER_SEED.to_csv(
    ALL5_ROOT/"FACETGATE_ALL5_LOCKED_TEST_TARGET_LABELS_PER_SEED.csv",index=False
)

target_summary=[]
for lab in CFG["targets"]:
    z=TARGET_PER_SEED[TARGET_PER_SEED.target_label==lab]
    for metric in ["f1","average_precision"]:
        vals=z[metric].to_numpy(dtype=float)
        mean=float(vals.mean())
        sd=float(vals.std(ddof=1))
        half=T_CRIT_DF4*sd/math.sqrt(5)
        target_summary.append({
            "target_label":lab,
            "metric":metric,
            "mean":mean,
            "sample_sd":sd,
            "ci95_low":mean-half,
            "ci95_high":mean+half,
            "mean_pct":100*mean,
        })
pd.DataFrame(target_summary).to_csv(
    ALL5_ROOT/"FACETGATE_ALL5_LOCKED_TEST_TARGET_LABEL_MEAN_SD_CI.csv",index=False
)

# Headline metrics.
def S(metric):
    return SUMMARY.loc[SUMMARY.metric==metric].iloc[0]

headline=pd.DataFrame([
    {
        "scope":"5-task",
        "accuracy_mean_pct":S("5task_accuracy")["mean_pct"],
        "accuracy_sd_pct":S("5task_accuracy")["sample_sd_pct"],
        "macro_f1_mean_pct":S("5task_macro_f1")["mean_pct"],
        "macro_f1_sd_pct":S("5task_macro_f1")["sample_sd_pct"],
    },
    {
        "scope":"Target",
        "accuracy_mean_pct":S("target_exact_accuracy")["mean_pct"],
        "accuracy_sd_pct":S("target_exact_accuracy")["sample_sd_pct"],
        "macro_f1_mean_pct":S("target_macro_f1")["mean_pct"],
        "macro_f1_sd_pct":S("target_macro_f1")["sample_sd_pct"],
    },
    {
        "scope":"6-task",
        "accuracy_mean_pct":S("6task_accuracy")["mean_pct"],
        "accuracy_sd_pct":S("6task_accuracy")["sample_sd_pct"],
        "macro_f1_mean_pct":S("6task_macro_f1")["mean_pct"],
        "macro_f1_sd_pct":S("6task_macro_f1")["sample_sd_pct"],
    },
])
headline.to_csv(ALL5_ROOT/"FACETGATE_HEADLINE_5SEED_LOCKED_TEST.csv",index=False)

print("\nPER-SEED LOCKED TEST (%)")
print(percent.to_string(index=False,float_format=lambda x:f"{x:.3f}"))
print("\nFIVE-SEED MEAN / SD / 95% CI (%)")
print(
    SUMMARY[[
        "metric","mean_pct","sample_sd_pct","ci95_low_pct","ci95_high_pct"
    ]].to_string(index=False,float_format=lambda x:f"{x:.3f}")
)

In [ ]:
# Final manifest and result archive.
FINAL_MANIFEST={
    "schema":"M3Offend_BanglaBERT_CLIP_FacetGate_ALL5_FINAL_LOCKED_TEST_v1_0_1",
    "experiment_name":"FacetGate",
    "seeds":PUBLICATION_SEEDS,
    "protocol_sha256":EXPECTED_PROTOCOL_SHA256,
    "binding_sha256":FACET_BINDING_SHA256,
    "dataset_sha256":CFG["expected_sha256"],
    "development_freeze_sha256":DEVELOPMENT_FREEZE_SHA256,
    "all_five_development_complete_before_test":True,
    "development_test_forward_count":0,
    "locked_test_seed_passes":LOCKED_TEST_SEED_PASSES,
    "locked_test_rows_per_seed":932,
    "test_used_for_checkpoint_selection":False,
    "test_used_for_threshold_selection":False,
    "test_used_for_numerical_repair":False,
}
(ALL5_ROOT/"FINAL_FACETGATE_MANIFEST.json").write_text(
    json.dumps(FINAL_MANIFEST,indent=2),encoding="utf-8"
)

# Hash artifacts before packaging.
hash_rows=[]
for p in sorted(ALL5_ROOT.rglob("*")):
    if p.is_file():
        hash_rows.append({
            "relative_path":str(p.relative_to(ALL5_ROOT)),
            "bytes":int(p.stat().st_size),
            "sha256":sha256_file(p),
        })
pd.DataFrame(hash_rows).to_csv(
    ALL5_ROOT/"FINAL_ARTIFACT_SHA256_MANIFEST.csv",index=False
)

archive_base=Path(
    "/kaggle/working/"
    "M3Offend_9318_BanglaBERT_CLIP_FacetGate_ALL5_FINAL_LOCKED_TEST_RESULTS"
)
zip_path=Path(str(archive_base)+".zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(archive_base),"zip",ALL5_ROOT)

print("\n"+"="*110)
print("FACETGATE ALL-5 + FINAL LOCKED TEST COMPLETE")
print("="*110)
print("Seeds:",PUBLICATION_SEEDS)
print("Development TEST forwards:",0)
print("Final locked TEST seed passes:",LOCKED_TEST_SEED_PASSES)
print("Freeze SHA-256:",DEVELOPMENT_FREEZE_SHA256)
print("Final ZIP:",zip_path)
print("Final ZIP SHA-256:",sha256_file(zip_path))